# 66B · Post-E50 — Cross-Frame Registration

**Objetivo:** evaluar si Sagittal T1, Sagittal T2 y los clusters de orientación de Axial T2 —que
Notebook 66 encontró con `FrameOfReferenceUID` distintos y sin objeto DICOM de Registration
explícito— pueden relacionarse mediante una transformación rígida reproducible y **validable**,
sin asumir que sus coordenadas DICOM son directamente equivalentes.

## Por qué se necesita esta fase

Notebook 66 dejó:

- Series discovery: PASS
- Within-series DICOM geometry: PASS
- Pixel ↔ patient transforms: PASS
- Privacy audit: PASS
- Axial T2: 5 clusters de orientación reales
- Sagittal T1, Sagittal T2, Axial T2: `FrameOfReferenceUID` distintos, sin registro DICOM explícito
- Cross-series spatial relationship: **PARTIAL**

El requisito técnico resultante es `CROSS_FRAME_REGISTRATION`, que este notebook aborda.

## Alcance

**SÍ:** reconstrucción física correcta de volúmenes vía SimpleITK, registración rígida (6 DOF)
Sagittal T1 → Sagittal T2 con Mattes Mutual Information, validación independiente mediante
landmarks heurísticos (no clínicos), re-detección de los orientation clusters de Axial T2,
intento de registración rígida por cluster contra Sagittal T2, cadena de transformación
auditable con round-trip numérico.

**NO:** entrena redes neuronales, no implementa Notebook 67 (naming de niveles), no hace
deformable registration, no modifica código productivo ni `AUTOMATIC_DISC_LOCALIZATION_VALIDATED`,
no afirma corrección anatómica solo porque el optimizador convergió.

## Principio metodológico

Se distinguen explícitamente tres conceptos en todo el notebook:

1. **DICOM native geometry** — lo que el header DICOM declara, por serie.
2. **Estimated registration transform** — la salida numérica de un optimizador iterativo.
3. **Validated spatial correspondence** — solo cuando existe evidencia independiente del
   optimizador (landmarks, antes/después) que corrobora la transformación.

`REGISTRATION SUCCESS` (el optimizador terminó sin error) **no implica**
`REGISTRATION VALIDATION` (evidencia independiente de que el resultado es correcto).

## Decisión de diseño: sin paquete `lib/` externo

Igual que en Notebook 66, todas las funciones se definen inline en este notebook por las mismas
razones (autocontención, auditoría de un solo archivo, sin código productivo nuevo fuera de
`ai_service/`).

Rama: `research/post-e50-cross-frame-registration` · Rama madre: `research/post-e50-series-pairing`


In [1]:
# --- Setup: repository root, allowed write scope, git identity, privacy helpers ---
import hashlib
import io
import json
import math
import os
import subprocess
import sys
import time
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk

EXECUTION_START = time.time()


def _find_repo_root(start: Path) -> Path:
    # Walk upward from the notebook location until we find repo markers.
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "ai_service").is_dir() and (candidate / "config").is_dir():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook location")


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = _find_repo_root(NOTEBOOK_DIR)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

ALLOWED_WRITE_ROOTS = [
    REPO_ROOT / "notebooks" / "post_e50",
    REPO_ROOT / "artifacts" / "post_e50",
    REPO_ROOT / "reports" / "post_e50",
]
REG_DIR = REPO_ROOT / "artifacts" / "post_e50" / "cross_frame_registration"
TRANSFORMS_DIR = REG_DIR / "transforms"
FIGURES_DIR = REG_DIR / "figures"
REPORT_DIR = REPO_ROOT / "reports" / "post_e50"

warnings: list[str] = []
limitations: list[str] = []

FORBIDDEN_IDENTIFIER_FIELDS = (
    "PatientName", "PatientID", "AccessionNumber",
    "StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID", "InstitutionName",
)


def safe_write_text(path: Path, content: str) -> None:
    path = path.resolve()
    if not any(str(path).startswith(str(root.resolve())) for root in ALLOWED_WRITE_ROOTS):
        raise RuntimeError(f"Refusing to write outside allowed Post-E50 trees: {path}")
    forbidden_values = getattr(safe_write_text, "_forbidden_values", set())
    for value in forbidden_values:
        if value and value in content:
            raise RuntimeError(f"Refusing to persist forbidden identifier value into {path.name}")
    if "C:\\Users\\" in content or "/Users/" in content:
        raise RuntimeError(f"Refusing to persist a local filesystem path into {path.name}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def opaque_id(raw_uid: str) -> str:
    return hashlib.sha256(str(raw_uid).encode("utf-8")).hexdigest()[:12]


def run_git(*args: str) -> str:
    result = subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
    return result.stdout.strip()


GIT_BRANCH = run_git("branch", "--show-current")
GIT_COMMIT = run_git("rev-parse", "HEAD")
GIT_STATUS_PORCELAIN = run_git("status", "--porcelain")
GENERATED_AT = datetime.now(timezone.utc).isoformat()

print("REPO_ROOT (relative label only, no absolute path persisted):", REPO_ROOT.name)
print("GIT_BRANCH:", GIT_BRANCH)
print("GIT_COMMIT:", GIT_COMMIT)
print("GENERATED_AT:", GENERATED_AT)
print("Working tree clean:", GIT_STATUS_PORCELAIN == "")
print("SimpleITK version:", sitk.Version_VersionString())


REPO_ROOT (relative label only, no absolute path persisted): post-e50-baseline-inventory-cce171
GIT_BRANCH: research/post-e50-cross-frame-registration
GIT_COMMIT: 17f59ae186b43db0610aed85301b76b7cf3f358f
GENERATED_AT: 2026-08-18T00:46:32.729105+00:00
Working tree clean: False
SimpleITK version: 2.5.5


## 1. Fuente de datos DICOM (read-only, misma metodología que Notebook 66)

Mismo ZIP, misma búsqueda dinámica ascendente, sin copiar ni extraer permanentemente al repo.


In [2]:
def _redact_relative(path: Path, anchor_name: str = "PFI_MVP_Juntos") -> str:
    parts = path.resolve().parts
    masked_name = f"<{opaque_id(path.name)}>{path.suffix}"
    if anchor_name in parts:
        idx = parts.index(anchor_name)
        dir_parts = list(parts[idx:-1])
        return "/".join([*dir_parts, masked_name])
    return masked_name


def discover_dicom_source() -> tuple[Path | None, str]:
    env_value = os.environ.get("PFI_POST_E50_DICOM_STUDY")
    if env_value:
        candidate = Path(env_value)
        if candidate.exists():
            return candidate, "env:PFI_POST_E50_DICOM_STUDY"
        warnings.append("PFI_POST_E50_DICOM_STUDY is set but does not exist on disk.")

    anchor = None
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if parent.name == "PFI_MVP_Juntos":
            anchor = parent
            break
    if anchor is None:
        return None, "not_found:no_PFI_MVP_Juntos_ancestor"

    sibling_names = ["PFI_RM_Lumbar_Final", "PFI_RM_Lumbar_Profesores"]
    candidates = []
    for name in sibling_names:
        sibling = anchor / name
        test_data = sibling / "test-data"
        if test_data.is_dir():
            candidates.extend(sorted(test_data.glob("*.zip")))

    if not candidates:
        return None, "not_found:no_test_data_zip_in_siblings"

    expected_sha = "1C058033AADAF9C72AF8F1B5D85DBBBDCB7706FDDA50B74537CE71A55B2227B1".lower()
    for cand in candidates:
        digest = hashlib.sha256()
        with cand.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() == expected_sha:
            return cand, f"sibling_search:{_redact_relative(cand)}"

    return candidates[0], f"sibling_search_no_hash_match:{_redact_relative(candidates[0])}"


DICOM_SOURCE_PATH, DICOM_SOURCE_METHOD = discover_dicom_source()
REAL_DICOM_USED = DICOM_SOURCE_PATH is not None
print("DICOM source found:", REAL_DICOM_USED)
print("Discovery method:", DICOM_SOURCE_METHOD)

EXPECTED_STUDY_ZIP_SHA256 = "1C058033AADAF9C72AF8F1B5D85DBBBDCB7706FDDA50B74537CE71A55B2227B1".lower()
dicom_zip_sha256 = None
if REAL_DICOM_USED and DICOM_SOURCE_PATH.is_file():
    digest = hashlib.sha256()
    with DICOM_SOURCE_PATH.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    dicom_zip_sha256 = digest.hexdigest()
    print("SHA-256:", dicom_zip_sha256)
    print("Matches historically expected study hash:", dicom_zip_sha256 == EXPECTED_STUDY_ZIP_SHA256)
    if dicom_zip_sha256 != EXPECTED_STUDY_ZIP_SHA256:
        warnings.append("Located DICOM zip SHA-256 does not match the historically expected value; treated as a different dataset, not corruption.")
elif not REAL_DICOM_USED:
    warnings.append("No real DICOM study located; PFI_POST_E50_DICOM_STUDY not set and sibling search failed.")


DICOM source found: True
Discovery method: sibling_search:PFI_MVP_Juntos/PFI_RM_Lumbar_Final/test-data/<c40f992f99ed>.zip
SHA-256: 1c058033aadaf9c72af8f1b5d85dbbbdcb7706fdda50b74537ce71a55b2227b1
Matches historically expected study hash: True


## 2. Lectura DICOM en memoria (idéntica política que Notebook 66)

Nunca se escribe un `.dcm` ni un array de píxeles dentro del repositorio.


In [3]:
@dataclass
class DicomInstance:
    series_folder: str
    entry_name: str
    dataset: "pydicom.dataset.FileDataset"


def _iter_dicom_entries(source: Path):
    if source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as zf:
            for name in zf.namelist():
                if name.endswith(".dcm"):
                    with zf.open(name) as handle:
                        data = handle.read()
                    parts = name.split("/")
                    series_folder = parts[1] if len(parts) > 1 else "unknown"
                    yield series_folder, name, data
    elif source.is_dir():
        for path in source.rglob("*.dcm"):
            series_folder = path.parent.name
            yield series_folder, str(path.relative_to(source)), path.read_bytes()


def load_instances_metadata_only(source: Path) -> list[DicomInstance]:
    out = []
    for series_folder, entry_name, data in _iter_dicom_entries(source):
        ds = pydicom.dcmread(io.BytesIO(data), stop_before_pixels=True)
        out.append(DicomInstance(series_folder, entry_name, ds))
    return out


def load_pixel_array_for_entry(source: Path, entry_name: str) -> np.ndarray:
    if source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as zf:
            with zf.open(entry_name) as handle:
                data = handle.read()
        ds = pydicom.dcmread(io.BytesIO(data))
    else:
        ds = pydicom.dcmread(source / entry_name)
    return ds.pixel_array.astype(np.float32)


if REAL_DICOM_USED:
    instances = load_instances_metadata_only(DICOM_SOURCE_PATH)
    print("Total DICOM instances found:", len(instances))
else:
    instances = []
    print("No instances loaded (no real DICOM source).")


Total DICOM instances found: 48


## 3. Geometría DICOM (misma convención verificada que Notebook 66)

`row_cosines = IOP[:3]` (dirección al aumentar índice de columna), `column_cosines = IOP[3:]`
(dirección al aumentar índice de fila). `patient = IPP + col*PixelSpacing[1]*row_cosines +
row*PixelSpacing[0]*column_cosines`. `normal = cross(row_cosines, column_cosines)`.


In [4]:
def row_column_cosines(iop) -> tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(iop, dtype=np.float64)
    return arr[:3], arr[3:]


def plane_normal(row_cos: np.ndarray, col_cos: np.ndarray) -> np.ndarray:
    return np.cross(row_cos, col_cos)


def pixel_to_patient_xyz(image_position, image_orientation, pixel_spacing, row, col) -> np.ndarray:
    ipp = np.asarray(image_position, dtype=np.float64)
    row_cos, col_cos = row_column_cosines(image_orientation)
    d_row, d_col = float(pixel_spacing[0]), float(pixel_spacing[1])
    return ipp + col * d_col * row_cos + row * d_row * col_cos


def angle_between_normals_deg(n1: np.ndarray, n2: np.ndarray) -> float:
    cos_a = float(np.clip(abs(np.dot(n1, n2)) / (np.linalg.norm(n1) * np.linalg.norm(n2)), -1.0, 1.0))
    return float(np.degrees(np.arccos(cos_a)))


print("Geometry primitives redefined independently (self-contained notebook, same formulas as Notebook 66).")


Geometry primitives redefined independently (self-contained notebook, same formulas as Notebook 66).


## 4. Registration DICOM search (read-only, re-verificación)

Se repite la búsqueda read-only de objetos de Spatial/Deformable Registration DICOM, igual que
en Notebook 66 (2b), como parte de la re-verificación de este notebook.


In [5]:
REGISTRATION_SOP_CLASS_UIDS = {
    "1.2.840.10008.5.1.4.1.1.66.1": "spatial_registration_storage",
    "1.2.840.10008.5.1.4.1.1.66.3": "deformable_spatial_registration_storage",
}
registration_objects_found = []
if REAL_DICOM_USED:
    for inst in instances:
        ds = inst.dataset
        sop_class = str(getattr(ds, "SOPClassUID", ""))
        has_reg_seq = hasattr(ds, "RegistrationSequence") or hasattr(ds, "DeformableRegistrationSequence")
        if sop_class in REGISTRATION_SOP_CLASS_UIDS or has_reg_seq:
            registration_objects_found.append({"series_folder": inst.series_folder})

print("Registration objects found:", len(registration_objects_found))
if not registration_objects_found:
    print("Confirmed (again): no explicit DICOM Spatial/Deformable Registration object in this study.")


Registration objects found: 0
Confirmed (again): no explicit DICOM Spatial/Deformable Registration object in this study.


## 5. Series discovery y clasificación (re-detectada desde los DICOM, misma heurística)


In [6]:
def sanitize_series_description(value) -> str:
    if value is None:
        return ""
    return str(value)[:64]


def classify_series_role(sample_ds, description_sanitized: str) -> tuple[str, float]:
    if not hasattr(sample_ds, "ImageOrientationPatient") or sample_ds.ImageOrientationPatient is None:
        return "unknown", 0.0
    row_cos, col_cos = row_column_cosines(sample_ds.ImageOrientationPatient)
    normal = plane_normal(row_cos, col_cos)
    dominant_axis = int(np.argmax(np.abs(normal)))
    geometric_family = "sagittal" if dominant_axis == 0 else ("axial" if dominant_axis == 2 else "other")
    desc_lower = description_sanitized.lower()
    weighting = "t1" if "t1" in desc_lower else ("t2" if "t2" in desc_lower else None)
    orientation_score = 0.5 if geometric_family in ("sagittal", "axial") else 0.1
    text_score = 0.3 if weighting else 0.0
    if geometric_family == "sagittal" and weighting == "t1":
        role = "sagittal_t1"
    elif geometric_family == "sagittal" and weighting == "t2":
        role = "sagittal_t2"
    elif geometric_family == "axial" and weighting == "t2":
        role = "axial_t2"
    else:
        role = "other"
    return role, float(min(1.0, orientation_score + text_score))


series_geometry: dict[str, dict] = {}
series_rows = []
STUDY_OPAQUE_ID = "unknown_study"

if REAL_DICOM_USED:
    study_uid_raw = None
    by_series: dict[str, list[DicomInstance]] = {}
    for inst in instances:
        sid = str(inst.dataset.SeriesInstanceUID)
        by_series.setdefault(sid, []).append(inst)
        if study_uid_raw is None and hasattr(inst.dataset, "StudyInstanceUID"):
            study_uid_raw = str(inst.dataset.StudyInstanceUID)
    STUDY_OPAQUE_ID = opaque_id(study_uid_raw) if study_uid_raw else "unknown_study"

    for series_uid, insts in by_series.items():
        series_opaque = opaque_id(series_uid)
        sample = insts[0].dataset
        description_sanitized = sanitize_series_description(getattr(sample, "SeriesDescription", None))
        role, confidence = classify_series_role(sample, description_sanitized)
        for_uid = getattr(sample, "FrameOfReferenceUID", None)
        series_geometry[series_opaque] = {
            "for_uid_opaque": opaque_id(for_uid) if for_uid else None,
            "instances": insts,
            "sample_ds": sample,
            "role": role,
        }
        series_rows.append({
            "series_opaque_id": series_opaque, "candidate_role": role,
            "role_confidence": confidence, "slice_count": len(insts),
        })

series_inventory = pd.DataFrame(series_rows)
series_inventory


,series_opaque_id,candidate_role,role_confidence,slice_count
0,28ea4937243f,sagittal_t1,0.8,12
1,b65eca7cfbbc,axial_t2,0.8,24
2,25aa08338722,sagittal_t2,0.8,12


## 6. Orden físico de slices por serie (idéntico criterio que Notebook 66)


In [7]:
def order_series_physically(series_opaque_id: str) -> list[tuple[float, "DicomInstance"]]:
    g = series_geometry[series_opaque_id]
    sample = g["sample_ds"]
    row_cos, col_cos = row_column_cosines(sample.ImageOrientationPatient)
    normal = plane_normal(row_cos, col_cos)
    scored = []
    for inst in g["instances"]:
        ipp = np.asarray(inst.dataset.ImagePositionPatient, dtype=np.float64)
        scored.append((float(np.dot(ipp, normal)), inst))
    scored.sort(key=lambda item: item[0])
    return scored


slice_ordering = {sid: order_series_physically(sid) for sid in series_geometry} if REAL_DICOM_USED else {}

sag_t1_ids = series_inventory.loc[series_inventory["candidate_role"] == "sagittal_t1", "series_opaque_id"].tolist() if len(series_inventory) else []
sag_t2_ids = series_inventory.loc[series_inventory["candidate_role"] == "sagittal_t2", "series_opaque_id"].tolist() if len(series_inventory) else []
axial_ids = series_inventory.loc[series_inventory["candidate_role"] == "axial_t2", "series_opaque_id"].tolist() if len(series_inventory) else []
print("sagittal_t1:", sag_t1_ids)
print("sagittal_t2:", sag_t2_ids)
print("axial_t2:", axial_ids)


sagittal_t1: ['28ea4937243f']
sagittal_t2: ['25aa08338722']
axial_t2: ['b65eca7cfbbc']


## 7. Axial orientation clustering (re-detección, consistency check contra Notebook 66)

Se re-detectan los clusters de orientación de Axial T2 con la misma metodología (union-find
sobre distancia angular entre normales, tolerancia `1.0°`), **sin hardcodear** los conteos
`[6, 4, 4, 5, 5]` que Notebook 66 encontró — se comparan como consistency check después.


In [8]:
ORIENTATION_CLUSTER_ANGLE_TOLERANCE_DEG = 1.0


def cluster_series_by_orientation(series_opaque_id: str, tolerance_deg: float = ORIENTATION_CLUSTER_ANGLE_TOLERANCE_DEG) -> list[dict]:
    g = series_geometry[series_opaque_id]
    per_slice = []
    for inst in g["instances"]:
        ds = inst.dataset
        row_cos, col_cos = row_column_cosines(ds.ImageOrientationPatient)
        normal = plane_normal(row_cos, col_cos)
        normal = normal / np.linalg.norm(normal)
        per_slice.append({"inst": inst, "normal": normal, "ipp": np.asarray(ds.ImagePositionPatient, dtype=np.float64)})

    n = len(per_slice)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[ry] = rx

    for i in range(n):
        for j in range(i + 1, n):
            if angle_between_normals_deg(per_slice[i]["normal"], per_slice[j]["normal"]) <= tolerance_deg:
                union(i, j)

    groups: dict[int, list[int]] = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)

    clusters = []
    for cluster_id, (_, indices) in enumerate(sorted(groups.items(), key=lambda kv: kv[0])):
        members = [per_slice[i] for i in indices]
        mean_normal = np.mean([m["normal"] for m in members], axis=0)
        mean_normal = mean_normal / np.linalg.norm(mean_normal)
        ordered = sorted(members, key=lambda m: float(np.dot(m["ipp"], mean_normal)))
        clusters.append({
            "cluster_id": cluster_id,
            "slice_count": len(members),
            "mean_normal": mean_normal,
            "ordered_instances": ordered,
        })
    return clusters


axial_clusters_by_series: dict[str, list[dict]] = {}
if REAL_DICOM_USED:
    for sid in axial_ids:
        axial_clusters_by_series[sid] = cluster_series_by_orientation(sid)

axial_cluster_sizes = []
for sid, clusters in axial_clusters_by_series.items():
    axial_cluster_sizes = sorted(c["slice_count"] for c in clusters)

print("Axial orientation clusters re-detected:", len(axial_cluster_sizes), "sizes:", axial_cluster_sizes)

NOTEBOOK_66_CLUSTER_SIZES_ARTIFACT = REPO_ROOT / "artifacts" / "post_e50" / "series_pairing" / "post_e50_axial_orientation_clusters.csv"
if NOTEBOOK_66_CLUSTER_SIZES_ARTIFACT.is_file():
    nb66_clusters = pd.read_csv(NOTEBOOK_66_CLUSTER_SIZES_ARTIFACT)
    nb66_sizes = sorted(nb66_clusters["slice_count"].tolist())
    consistency_match = (nb66_sizes == axial_cluster_sizes)
    print("Notebook 66 artifact cluster sizes:", nb66_sizes)
    print("Consistency check (re-detection matches Notebook 66 artifact):", consistency_match)
    if not consistency_match:
        warnings.append(f"Axial cluster re-detection {axial_cluster_sizes} differs from Notebook 66 artifact {nb66_sizes}.")
else:
    warnings.append("Notebook 66 axial orientation cluster artifact not found; consistency check skipped.")
    print("Notebook 66 cluster artifact not found on disk; skipping consistency check.")


Axial orientation clusters re-detected: 5 sizes: [4, 4, 5, 5, 6]
Notebook 66 artifact cluster sizes: [4, 4, 5, 5, 6]
Consistency check (re-detection matches Notebook 66 artifact): True


## 8. Reconstrucción de volumen SimpleITK — Sagittal T2 (reference space)

Se usa **Sagittal T2** como `fixed`/reference space por decisión de ingeniería experimental
(ya es central en el pipeline de segmentación/localización, y Notebook 66 confirmó su geometría
intraserie válida) — **no** se afirma superioridad clínica.

La reconstrucción **no** usa `np.stack` ordenado por nombre de archivo. Se construye
explícitamente `origin`, `spacing` y `direction` a partir de la geometría física validada:

- `origin` = `ImagePositionPatient` del primer slice en orden físico.
- `spacing` = `(PixelSpacing[1], PixelSpacing[0], slice_spacing)` (orden ITK x,y,z).
- `direction` = columnas `[row_cosines, column_cosines, slice_direction]`, donde
  `slice_direction` es el vector normalizado real entre el primer y segundo slice físico (no
  simplemente el `normal` de un solo slice, para capturar el sentido real del stack).


In [9]:
def build_sitk_volume(series_opaque_id: str) -> tuple["sitk.Image", dict]:
    ordered = slice_ordering[series_opaque_id]
    first_ds = ordered[0][1].dataset
    row_cos, col_cos = row_column_cosines(first_ds.ImageOrientationPatient)

    if len(ordered) > 1:
        p0 = np.asarray(ordered[0][1].dataset.ImagePositionPatient, dtype=np.float64)
        p1 = np.asarray(ordered[1][1].dataset.ImagePositionPatient, dtype=np.float64)
        slice_vec = p1 - p0
        slice_spacing = float(np.linalg.norm(slice_vec))
        slice_dir = slice_vec / slice_spacing if slice_spacing > 0 else plane_normal(row_cos, col_cos)
    else:
        slice_dir = plane_normal(row_cos, col_cos)
        slice_spacing = float(getattr(first_ds, "SpacingBetweenSlices", getattr(first_ds, "SliceThickness", 1.0)))

    pixel_spacing = first_ds.PixelSpacing
    origin = np.asarray(ordered[0][1].dataset.ImagePositionPatient, dtype=np.float64)

    arrays = [load_pixel_array_for_entry(DICOM_SOURCE_PATH, inst.entry_name) for _, inst in ordered]
    volume = np.stack(arrays, axis=0).astype(np.float32)

    img = sitk.GetImageFromArray(volume)
    img.SetOrigin(tuple(origin))
    img.SetSpacing((float(pixel_spacing[1]), float(pixel_spacing[0]), slice_spacing))
    direction_matrix = np.column_stack([row_cos, col_cos, slice_dir])
    img.SetDirection(tuple(direction_matrix.flatten()))

    meta = {
        "series_opaque_id": series_opaque_id,
        "origin": origin.tolist(),
        "spacing": [float(pixel_spacing[1]), float(pixel_spacing[0]), slice_spacing],
        "direction": direction_matrix.flatten().tolist(),
        "size": list(volume.shape[::-1]),
        "row_cosines": row_cos.tolist(),
        "column_cosines": col_cos.tolist(),
        "slice_direction": slice_dir.tolist(),
    }
    return img, meta


sitk_volumes: dict[str, "sitk.Image"] = {}
sitk_volume_meta: dict[str, dict] = {}
DICOM_RECONSTRUCTION_OK = False

if REAL_DICOM_USED and sag_t2_ids and sag_t1_ids:
    for sid in [sag_t2_ids[0], sag_t1_ids[0]]:
        img, meta = build_sitk_volume(sid)
        sitk_volumes[sid] = img
        sitk_volume_meta[sid] = meta
    DICOM_RECONSTRUCTION_OK = True
    print("Reconstructed volumes for:", list(sitk_volumes.keys()))
    for sid, img in sitk_volumes.items():
        print(f"  {sid}: size={img.GetSize()} spacing={img.GetSpacing()} origin={img.GetOrigin()}")
else:
    warnings.append("Could not reconstruct SimpleITK volumes: missing sagittal_t1 or sagittal_t2 series.")

GATE_A_dicom_reconstruction = "PASS" if DICOM_RECONSTRUCTION_OK else "FAIL"
print("GATE_A_dicom_reconstruction:", GATE_A_dicom_reconstruction)


Reconstructed volumes for: ['25aa08338722', '28ea4937243f']
  25aa08338722: size=(512, 512, 12) spacing=(0.5859, 0.5859, 5.500014370535771) origin=(15.2846, -100.047, 184.573)
  28ea4937243f: size=(512, 512, 12) spacing=(0.5859, 0.5859, 5.500014370535771) origin=(15.2846, -100.047, 184.573)
GATE_A_dicom_reconstruction: PASS


## 9. GATE B — Paridad física SimpleITK vs Notebook 66

`TransformIndexToPhysicalPoint` de SimpleITK debe coincidir con `pixel_to_patient_xyz` de
Notebook 66 para puntos aleatorios dentro del volumen Sagittal T2 (serie de un solo cluster de
orientación, spacing regular). Objetivo: error `< 1e-3 mm`. **El umbral no se modifica
retroactivamente** aunque el resultado observado lo supere.

**Interpretación corregida de la causa raíz (segunda revisión).** La explicación anterior
("DICOM usa ~6 cifras significativas") ya fue corregida por ser una afirmación incorrecta sobre
el estándar DICOM. La revisión posterior (`REGULAR_GRID_APPROXIMATION_OF_SLIGHTLY_NONUNIFORM_
DICOM_SLICE_POSITIONS`) resultó **demasiado específica y no completamente demostrada**: el
residual escalar de posición medido en la Sección 9b (~0.0002 mm máximo) es un orden de magnitud
menor que el residual de paridad completa medido aquí (~0.00382 mm), por lo que la
no-uniformidad de posición **no explica por sí sola** el residual observado.

Causa raíz actualizada: **`REGULAR_GRID_APPROXIMATION_OF_SLICEWISE_DICOM_GEOMETRY`**, con cuatro
puntos explícitos:

1. SimpleITK representa el stack completo con una única grilla regular:
   `origin + spacing + direction` (un solo `origin`, un solo `spacing`, una sola matriz de
   `direction` para las 12 slices).
2. Los DICOM originales conservan geometría **slice-by-slice**: cada instancia tiene su propio
   `ImagePositionPatient` e `ImageOrientationPatient` independientes.
3. El residual escalar de posición observado (Sección 9b, `max≈0.0002 mm`) **no explica por sí
   mismo** el residual total de paridad (`max≈0.00382 mm`) — hay al menos un orden de magnitud
   de diferencia sin atribuir.
4. La variación de orientación slice-by-slice (`ImageOrientationPatient` no siendo idéntico en
   cada slice) es una **contribución posible** a esa diferencia no explicada, pero **no se
   declara como causa** hasta medirla explícitamente — ver Sección 9c.

Esto sigue sin ser una falla de las ecuaciones pixel↔patient de Notebook 66 (que operan slice por
slice, sobre la geometría nativa real, sin asumir grilla regular) — es, en el peor de los casos,
una limitación de aproximar una serie completa como un único `sitk.Image`, cuya fuente exacta
queda medida de forma analítica en la sección siguiente, no asumida.


In [10]:
GATE_B_parity_errors = []
if DICOM_RECONSTRUCTION_OK:
    ref_sid = sag_t2_ids[0]
    img = sitk_volumes[ref_sid]
    ordered = slice_ordering[ref_sid]
    rng = np.random.default_rng(2026)
    n_slices = len(ordered)
    rows, cols = int(ordered[0][1].dataset.Rows), int(ordered[0][1].dataset.Columns)

    for _ in range(30):
        k = int(rng.integers(0, n_slices))
        i = float(rng.uniform(0, cols - 1))
        j = float(rng.uniform(0, rows - 1))
        sitk_point = np.array(img.TransformContinuousIndexToPhysicalPoint((i, j, float(k))))

        ds_k = ordered[k][1].dataset
        nb66_point = pixel_to_patient_xyz(ds_k.ImagePositionPatient, ds_k.ImageOrientationPatient, ds_k.PixelSpacing, row=j, col=i)
        err = float(np.linalg.norm(sitk_point - nb66_point))
        GATE_B_parity_errors.append(err)

    max_parity_error_mm = max(GATE_B_parity_errors)
    GATE_B_orientation = "PASS" if max_parity_error_mm < 1e-3 else "FAIL"
    print(f"Max SimpleITK/Notebook-66 parity error: {max_parity_error_mm:.3e} mm over {len(GATE_B_parity_errors)} points")
else:
    max_parity_error_mm = None
    GATE_B_orientation = "FAIL"
    warnings.append("GATE B could not run: no reconstructed volumes.")

GATE_B_FAILURE_INTERPRETATION = "REGULAR_GRID_APPROXIMATION_OF_SLICEWISE_DICOM_GEOMETRY" if GATE_B_orientation == "FAIL" else None

print("GATE_B_orientation (SimpleITK physical-space parity):", GATE_B_orientation)
print("gate_b_failure_interpretation:", GATE_B_FAILURE_INTERPRETATION)
if GATE_B_FAILURE_INTERPRETATION:
    print("This failure does NOT demonstrate an error in Notebook 66's pixel<->patient equations;")
    print("it reflects the regular-grid approximation SimpleITK requires to represent a full series as one Image.")


Max SimpleITK/Notebook-66 parity error: 3.820e-03 mm over 30 points
GATE_B_orientation (SimpleITK physical-space parity): FAIL
gate_b_failure_interpretation: REGULAR_GRID_APPROXIMATION_OF_SLICEWISE_DICOM_GEOMETRY
This failure does NOT demonstrate an error in Notebook 66's pixel<->patient equations;
it reflects the regular-grid approximation SimpleITK requires to represent a full series as one Image.


## 9b. Auditoría del residual — `simpleitk_geometry_residuals`

Por cada slice de la serie de referencia (Sagittal T2), se compara la posición física real
(escalar proyectado sobre la normal del plano, **no** XYZ crudo -- se evita persistir
coordenadas absolutas de paciente) contra la posición que asumiría la grilla regular de
SimpleITK (`origin_scalar + slice_index * slice_spacing`). Solo se trabaja con posiciones
escalares relativas, consistente con la política de privacidad vigente.


In [11]:
simpleitk_geometry_residuals_rows = []
if DICOM_RECONSTRUCTION_OK:
    ref_sid = sag_t2_ids[0]
    ordered = slice_ordering[ref_sid]
    first_ds = ordered[0][1].dataset
    row_cos, col_cos = row_column_cosines(first_ds.ImageOrientationPatient)
    normal = plane_normal(row_cos, col_cos)
    normal = normal / np.linalg.norm(normal)

    origin_xyz = np.asarray(ordered[0][1].dataset.ImagePositionPatient, dtype=np.float64)
    origin_scalar = float(np.dot(origin_xyz, normal))
    meta = sitk_volume_meta[ref_sid]
    slice_spacing = meta["spacing"][2]
    slice_dir = np.asarray(meta["slice_direction"], dtype=np.float64)
    # Projection of the regular-grid direction onto the same normal used for physical ordering
    # (they coincide up to sign for a clean single-cluster sagittal series).
    grid_step_scalar = slice_spacing * float(np.dot(slice_dir, normal))

    for k, (_, inst) in enumerate(ordered):
        ipp = np.asarray(inst.dataset.ImagePositionPatient, dtype=np.float64)
        dicom_position_scalar_mm = float(np.dot(ipp, normal))
        regular_grid_position_scalar_mm = origin_scalar + k * grid_step_scalar
        simpleitk_geometry_residuals_rows.append({
            "slice_index": k,
            "dicom_position_scalar_mm": dicom_position_scalar_mm,
            "regular_grid_position_scalar_mm": regular_grid_position_scalar_mm,
            "absolute_residual_mm": abs(dicom_position_scalar_mm - regular_grid_position_scalar_mm),
        })

simpleitk_geometry_residuals = pd.DataFrame(simpleitk_geometry_residuals_rows)
if len(simpleitk_geometry_residuals):
    max_residual_mm = float(simpleitk_geometry_residuals["absolute_residual_mm"].max())
    mean_residual_mm = float(simpleitk_geometry_residuals["absolute_residual_mm"].mean())
    median_residual_mm = float(simpleitk_geometry_residuals["absolute_residual_mm"].median())
    median_slice_spacing_mm_ref = float(np.median(np.abs(np.diff(simpleitk_geometry_residuals["dicom_position_scalar_mm"]))))
    max_residual_relative_to_slice_spacing = max_residual_mm / median_slice_spacing_mm_ref if median_slice_spacing_mm_ref else None
else:
    max_residual_mm = mean_residual_mm = median_residual_mm = None
    median_slice_spacing_mm_ref = None
    max_residual_relative_to_slice_spacing = None

print(simpleitk_geometry_residuals)
print()
print("max_residual_mm:", max_residual_mm)
print("mean_residual_mm:", mean_residual_mm)
print("median_residual_mm:", median_residual_mm)
print("median_slice_spacing_mm:", median_slice_spacing_mm_ref)
print("max_residual_relative_to_slice_spacing (engineering metric, not clinical):", max_residual_relative_to_slice_spacing)


    slice_index  dicom_position_scalar_mm  regular_grid_position_scalar_mm  \
0             0                -22.710492                       -22.710492   
1             1                -17.210477                       -17.210477   
2             2                -11.710483                       -11.710463   
3             3                 -6.210488                        -6.210448   
4             4                 -0.710494                        -0.710434   
5             5                  4.789505                         4.789580   
6             6                 10.289450                        10.289595   
7             7                 15.789494                        15.789609   
8             8                 21.289538                        21.289623   
9             9                 26.789483                        26.789638   
10           10                 32.289522                        32.289652   
11           11                 37.789467                       

## 9c. Variación de orientación slice-by-slice (analítico, sin cambiar el experimento)

Medición puramente analítica sobre metadata ya cargada (no repite ni modifica la reconstrucción
ni la registración): para Sagittal T2 y Sagittal T1, se calcula el ángulo entre la normal de
plano de cada slice y la normal del primer slice físico de la misma serie
(`slice_orientation_angle_vs_first_deg`). Esto mide directamente si la Sección 9's supuesto de
"una sola orientación para todo el stack" es válido, en vez de asumirlo.


In [12]:
def slice_orientation_deviation(series_opaque_id: str) -> pd.DataFrame:
    ordered = slice_ordering[series_opaque_id]
    first_ds = ordered[0][1].dataset
    row_cos0, col_cos0 = row_column_cosines(first_ds.ImageOrientationPatient)
    normal0 = plane_normal(row_cos0, col_cos0)
    normal0 = normal0 / np.linalg.norm(normal0)

    rows = []
    for k, (_, inst) in enumerate(ordered):
        ds = inst.dataset
        row_cos, col_cos = row_column_cosines(ds.ImageOrientationPatient)
        normal_k = plane_normal(row_cos, col_cos)
        normal_k = normal_k / np.linalg.norm(normal_k)
        rows.append({
            "slice_index": k,
            "slice_orientation_angle_vs_first_deg": angle_between_normals_deg(normal_k, normal0),
        })
    return pd.DataFrame(rows)


slice_orientation_deviation_by_series = {}
if DICOM_RECONSTRUCTION_OK:
    for role, sid in [("sagittal_t2", sag_t2_ids[0]), ("sagittal_t1", sag_t1_ids[0])]:
        df = slice_orientation_deviation(sid)
        slice_orientation_deviation_by_series[role] = df
        max_dev = float(df["slice_orientation_angle_vs_first_deg"].max())
        mean_dev = float(df["slice_orientation_angle_vs_first_deg"].mean())
        print(f"{role}: max_orientation_deviation_deg={max_dev:.6f}  mean_orientation_deviation_deg={mean_dev:.6f}")
        print(df)
        print()
else:
    print("Skipped: DICOM reconstruction unavailable.")


sagittal_t2: max_orientation_deviation_deg=0.000000  mean_orientation_deviation_deg=0.000000
    slice_index  slice_orientation_angle_vs_first_deg
0             0                                   0.0
1             1                                   0.0
2             2                                   0.0
3             3                                   0.0
4             4                                   0.0
5             5                                   0.0
6             6                                   0.0
7             7                                   0.0
8             8                                   0.0
9             9                                   0.0
10           10                                   0.0
11           11                                   0.0

sagittal_t1: max_orientation_deviation_deg=0.000000  mean_orientation_deviation_deg=0.000000
    slice_index  slice_orientation_angle_vs_first_deg
0             0                                   0.0
1  

## 10. Sagittal T1 → Sagittal T2: registración rígida

`fixed = Sagittal T2`, `moving = Sagittal T1`. Notebook 66 encontró `orientation_angle_deg≈0`,
`center_distance_mm≈0`, `physical_overlap_estimate≈1`, pero `FrameOfReferenceUID` distinto —
**no se asume identity**. El transform inicial se calcula por geometría
(`CenteredTransformInitializer`, modo `GEOMETRY`), no por identidad ciega.

- Transform: `Euler3DTransform` (rígido, 6 DOF: Tx,Ty,Tz,Rx,Ry,Rz)
- Métrica: Mattes Mutual Information (`numberOfHistogramBins=50`), sampling `RANDOM` al
  `20%`, `seed=2026` (reproducible)
- Multi-resolución: `shrinkFactors=[4,2,1]`, `smoothingSigmas=[2,1,0]` (unidades físicas)
- Optimizador: `RegularStepGradientDescent`, escalas automáticas por desplazamiento físico
  (`SetOptimizerScalesFromPhysicalShift`)


In [13]:
sagittal_registration_result = None
if DICOM_RECONSTRUCTION_OK:
    fixed_img = sitk.Cast(sitk_volumes[sag_t2_ids[0]], sitk.sitkFloat32)
    moving_img = sitk.Cast(sitk_volumes[sag_t1_ids[0]], sitk.sitkFloat32)

    initial_transform = sitk.CenteredTransformInitializer(
        fixed_img, moving_img, sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY,
    )

    method = sitk.ImageRegistrationMethod()
    NUMBER_OF_HISTOGRAM_BINS = 50
    SAMPLING_STRATEGY = "RANDOM"
    SAMPLING_PERCENTAGE = 0.20
    SAMPLING_SEED = 2026
    method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=NUMBER_OF_HISTOGRAM_BINS)
    method.SetMetricSamplingStrategy(method.RANDOM)
    method.SetMetricSamplingPercentage(SAMPLING_PERCENTAGE, seed=SAMPLING_SEED)
    method.SetInterpolator(sitk.sitkLinear)

    SHRINK_FACTORS = [4, 2, 1]
    SMOOTHING_SIGMAS = [2, 1, 0]
    method.SetShrinkFactorsPerLevel(SHRINK_FACTORS)
    method.SetSmoothingSigmasPerLevel(SMOOTHING_SIGMAS)
    method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    method.SetOptimizerAsRegularStepGradientDescent(
        learningRate=1.0, minStep=1e-4, numberOfIterations=200, gradientMagnitudeTolerance=1e-6,
    )
    method.SetOptimizerScalesFromPhysicalShift()
    method.SetInitialTransform(initial_transform, inPlace=False)

    # metric_before: evaluate the metric at the initial (geometry-based) transform, prior to optimization.
    metric_before = method.MetricEvaluate(fixed_img, moving_img)

    optimizer_error = None
    try:
        final_transform = method.Execute(fixed_img, moving_img)
        metric_after = method.GetMetricValue()
        optimizer_stop_condition = method.GetOptimizerStopConditionDescription()
        optimizer_iterations = method.GetOptimizerIteration()
    except RuntimeError as exc:
        optimizer_error = str(exc)
        final_transform = initial_transform
        metric_after = None
        optimizer_stop_condition = f"FAILED: {optimizer_error}"
        optimizer_iterations = None

    # Mattes MI in SimpleITK's ImageRegistrationMethod is minimized (more negative == higher MI == better).
    # "Improvement" therefore means metric_after < metric_before (more negative), not a larger raw value.
    if metric_after is not None:
        metric_improved = bool(metric_after < metric_before)
    else:
        metric_improved = False

    sagittal_registration_result = {
        "fixed_series": sag_t2_ids[0],
        "moving_series": sag_t1_ids[0],
        "metric_before": float(metric_before),
        "metric_after": float(metric_after) if metric_after is not None else None,
        "metric_improved_lower_is_better": metric_improved,
        "optimizer_error": optimizer_error,
        "optimizer_stop_condition": optimizer_stop_condition,
        "optimizer_iterations": optimizer_iterations,
        "final_transform": final_transform,
        "initial_transform": initial_transform,
    }
    print("metric_before (Mattes MI, minimized -> lower/more negative is better):", metric_before)
    print("metric_after:", metric_after)
    print("metric_improved:", metric_improved)
    print("optimizer_stop_condition:", optimizer_stop_condition)
else:
    warnings.append("Sagittal T1->T2 registration skipped: DICOM reconstruction unavailable.")

GATE_C_registration_optimization = "PASS" if (
    sagittal_registration_result is not None
    and sagittal_registration_result["optimizer_error"] is None
    and sagittal_registration_result["metric_after"] is not None
    and np.isfinite(sagittal_registration_result["metric_after"])
) else "FAIL"
print("GATE_C_registration_optimization:", GATE_C_registration_optimization)


metric_before (Mattes MI, minimized -> lower/more negative is better): -0.7821203706655426
metric_after: -1.05823428347514
metric_improved: True
optimizer_stop_condition: RegularStepGradientDescentOptimizerv4: Step too small after 23 iterations. Current step (6.10352e-05) is less than minimum step (0.0001).
GATE_C_registration_optimization: PASS


## 11. Serialización auditable del transform + sanity gates

Se serializa el transform como JSON (traducción, rotación, matriz 4x4, centro, disponibilidad de
inversa) en vez de guardar solo un objeto Python opaco. Se guarda también el transform nativo de
SimpleITK en `transforms/` (no contiene metadata sensible: son solo parámetros numéricos).

Umbrales de ingeniería (NO clínicos) para detectar soluciones sospechosas, dado que Notebook 66
encontró las dos series sagitales muy compatibles numéricamente:

- `translation magnitude > 30 mm` → WARNING (no rechazo automático)
- `rotation magnitude > 15 deg` → WARNING (no rechazo automático)


In [14]:
def serialize_euler3d_transform(transform: "sitk.Euler3DTransform") -> dict:
    params = transform.GetParameters()  # (rx, ry, rz, tx, ty, tz) in radians/mm
    rx, ry, rz, tx, ty, tz = params
    rotation_deg = [float(np.degrees(rx)), float(np.degrees(ry)), float(np.degrees(rz))]
    translation_mm = [float(tx), float(ty), float(tz)]
    matrix = np.array(transform.GetMatrix()).reshape(3, 3)
    center = np.array(transform.GetCenter())
    # Build 4x4 homogeneous matrix consistent with ITK's fixed-center rotation convention:
    # p' = M(p - c) + t + c
    m4 = np.eye(4)
    m4[:3, :3] = matrix
    m4[:3, 3] = matrix @ (-center) + np.array(translation_mm) + center
    try:
        inverse = transform.GetInverse()
        inverse_available = True
    except Exception:
        inverse_available = False

    translation_magnitude_mm = float(np.linalg.norm(translation_mm))
    rotation_magnitude_deg = float(np.linalg.norm(rotation_deg))
    return {
        "transform_type": "rigid_euler3d",
        "translation_xyz_mm": translation_mm,
        "rotation_parameters_rad": [float(rx), float(ry), float(rz)],
        "rotation_deg": rotation_deg,
        "transform_center_xyz": center.tolist(),
        "matrix_4x4": m4.tolist(),
        "inverse_available": inverse_available,
        "translation_magnitude_mm": translation_magnitude_mm,
        "rotation_magnitude_deg": rotation_magnitude_deg,
    }


sagittal_transform_json = None
sagittal_sanity_warnings = []
if sagittal_registration_result is not None:
    final_t = sagittal_registration_result["final_transform"]
    if isinstance(final_t, sitk.CompositeTransform):
        euler_component = sitk.Euler3DTransform(final_t.GetNthTransform(final_t.GetNumberOfTransforms() - 1))
    else:
        euler_component = sitk.Euler3DTransform(final_t)
    sagittal_transform_json = serialize_euler3d_transform(euler_component)

    if sagittal_transform_json["translation_magnitude_mm"] > 30.0:
        sagittal_sanity_warnings.append("translation_magnitude_exceeds_30mm_engineering_threshold")
    if sagittal_transform_json["rotation_magnitude_deg"] > 15.0:
        sagittal_sanity_warnings.append("rotation_magnitude_exceeds_15deg_engineering_threshold")

    sagittal_transform_json["sanity_warnings"] = sagittal_sanity_warnings
    warnings.extend(sagittal_sanity_warnings)

    sitk.WriteTransform(euler_component, str(TRANSFORMS_DIR / "sagittal_t1_to_t2_euler3d.tfm"))
    print(json.dumps(sagittal_transform_json, indent=2))
else:
    print("No sagittal transform to serialize.")


{
  "transform_type": "rigid_euler3d",
  "translation_xyz_mm": [
    0.5348534970416249,
    1.9595869218276107,
    -0.10288159070994625
  ],
  "rotation_parameters_rad": [
    -0.004520048135995739,
    0.0019043700884219881,
    -0.000432173595456761
  ],
  "rotation_deg": [
    -0.25897968138853056,
    0.10911236869753532,
    -0.024761723036666614
  ],
  "transform_center_xyz": [
    -5.4688070835000016,
    50.76933662999994,
    34.384231529999994
  ],
  "matrix_4x4": [
    [
      0.9999980895809255,
      0.0004321691671775392,
      0.00190632219470587,
      0.4473546836851696
    ],
    [
      -0.00044078060748880843,
      0.9999896912137637,
      0.0045192011083313504,
      1.802310490718277
    ],
    [
      -0.001904349483458572,
      -0.0045200327446184045,
      0.9999879713061715,
      0.11659655072921282
    ],
    [
      0.0,
      0.0,
      0.0,
      1.0
    ]
  ],
  "inverse_available": true,
  "translation_magnitude_mm": 2.033871625546331,
  "rotation_

## 12. Validación visual T1/T2 (no es la única validación)


In [15]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def mid_slice_2d(img: "sitk.Image") -> np.ndarray:
    arr = sitk.GetArrayFromImage(img)
    return arr[arr.shape[0] // 2]


def checkerboard(a: np.ndarray, b: np.ndarray, block: int = 20) -> np.ndarray:
    h, w = a.shape
    board = np.zeros((h, w), dtype=np.float32)
    for r in range(0, h, block):
        for c in range(0, w, block):
            use_a = ((r // block) + (c // block)) % 2 == 0
            src = a if use_a else b
            board[r:r + block, c:c + block] = src[r:r + block, c:c + block]
    return board


if sagittal_registration_result is not None:
    fixed_img = sitk.Cast(sitk_volumes[sag_t2_ids[0]], sitk.sitkFloat32)
    moving_img = sitk.Cast(sitk_volumes[sag_t1_ids[0]], sitk.sitkFloat32)
    resampled_after = sitk.Resample(moving_img, fixed_img, sagittal_registration_result["final_transform"], sitk.sitkLinear, 0.0, fixed_img.GetPixelID())
    resampled_before = sitk.Resample(moving_img, fixed_img, sagittal_registration_result["initial_transform"], sitk.sitkLinear, 0.0, fixed_img.GetPixelID())

    fixed_mid = mid_slice_2d(fixed_img)
    before_mid = mid_slice_2d(resampled_before)
    after_mid = mid_slice_2d(resampled_after)

    for name, arr in [("sagittal_registration_before", before_mid), ("sagittal_registration_after", after_mid)]:
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(arr, cmap="gray")
        ax.set_title(name)
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / f"{name}.png", dpi=120)
        plt.close(fig)

    def norm01(a):
        lo, hi = np.percentile(a, [1, 99])
        return np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)

    cb_before = checkerboard(norm01(fixed_mid), norm01(before_mid))
    cb_after = checkerboard(norm01(fixed_mid), norm01(after_mid))
    for name, arr in [("sagittal_checkerboard_before", cb_before), ("sagittal_checkerboard_after", cb_after)]:
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(arr, cmap="gray")
        ax.set_title(name)
        plt.tight_layout()
        fig.savefig(FIGURES_DIR / f"{name}.png", dpi=120)
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(np.abs(norm01(after_mid) - norm01(fixed_mid)), cmap="gray")
    ax.set_title("abs difference (fixed vs registered moving, normalized)")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "sagittal_difference_after.png", dpi=120)
    plt.close(fig)
    print("Visual validation figures written.")
else:
    print("Skipped: no sagittal registration result.")


Visual validation figures written.


## 13. Validación independiente (GATE D)

Se buscó primero si existen segmentaciones/ROIs ya disponibles para obtener landmarks sin
entrenar nada: no se encontró ninguna en este entorno de ejecución (ningún checkpoint de
segmentación está cargado ni corrido en este notebook, por diseño — ver Notebook 65).

`validation_source = "manual_research_landmarks"`: se usan 3 puntos por serie, elegidos por
**criterio puramente geométrico** (centro de imagen del slice físico al 15%/50%/85% del stack),
**NO** por criterio clínico ni anatómico específico. La correspondencia T1↔T2 asume rango físico
similar (misma región lumbar, mismo tipo de FOV), no una anotación experta. Esto es
explícitamente una validación de investigación, no una dependencia productiva, y solo se
persisten coordenadas relativas/anonimizadas necesarias para reproducibilidad (fracciones de
stack + XYZ en espacio paciente, sin UID).

Estos 3 puntos por serie se eligen **antes** de aplicar cualquier transform, por lo que la
comparación before/after es independiente del optimizador (el optimizador nunca ve estos puntos).

**Advertencia de circularidad metodológica, documentada explícitamente y no ocultada:** para el
par Sagittal T1/T2 específicamente, ambos series comparten casi la misma geometría numérica
(Notebook 66: `orientation_angle_deg≈0`, `center_distance_mm≈0`), y el landmark en cada serie usa
el mismo criterio geométrico (centro de imagen a la misma fracción de profundidad física). Por
lo tanto, `TRE_before` tiende a ser trivialmente pequeño **por construcción**, no porque el
`initial_transform` geométrico haya demostrado nada anatómico de forma independiente. Esta
validación es útil como chequeo direccional (¿el optimizador aleja o acerca estos puntos
heurísticos?) pero es una validación **débil**, no una confirmación anatómica fuerte. Se reporta
así de manera transparente en vez de presentarla como más sólida de lo que es.


In [16]:
def geometric_landmarks(series_opaque_id: str, fractions=(0.15, 0.5, 0.85)) -> list[dict]:
    ordered = slice_ordering[series_opaque_id]
    n = len(ordered)
    ds0 = ordered[0][1].dataset
    rows, cols = int(ds0.Rows), int(ds0.Columns)
    landmarks = []
    for frac in fractions:
        idx = int(round(frac * (n - 1)))
        s, inst = ordered[idx]
        ds = inst.dataset
        xyz = pixel_to_patient_xyz(ds.ImagePositionPatient, ds.ImageOrientationPatient, ds.PixelSpacing, rows / 2.0, cols / 2.0)
        landmarks.append({"fraction": frac, "slice_index_physical": idx, "patient_xyz": xyz.tolist()})
    return landmarks


VALIDATION_SOURCE = "unavailable"
tre_rows = []
if sagittal_registration_result is not None:
    t1_landmarks = geometric_landmarks(sag_t1_ids[0])
    t2_landmarks = geometric_landmarks(sag_t2_ids[0])
    VALIDATION_SOURCE = "manual_research_landmarks"

    final_inverse = sagittal_registration_result["final_transform"].GetInverse()
    initial_inverse = sagittal_registration_result["initial_transform"].GetInverse()

    for lm1, lm2 in zip(t1_landmarks, t2_landmarks):
        p1 = np.array(lm1["patient_xyz"])
        p2 = np.array(lm2["patient_xyz"])
        p1_before = np.array(initial_inverse.TransformPoint(p1.tolist()))
        p1_after = np.array(final_inverse.TransformPoint(p1.tolist()))
        tre_rows.append({
            "fraction": lm1["fraction"],
            "tre_before_mm": float(np.linalg.norm(p1_before - p2)),
            "tre_after_mm": float(np.linalg.norm(p1_after - p2)),
        })

tre_df = pd.DataFrame(tre_rows)
if len(tre_df):
    mean_tre_before = float(tre_df["tre_before_mm"].mean())
    median_tre_before = float(tre_df["tre_before_mm"].median())
    max_tre_before = float(tre_df["tre_before_mm"].max())
    mean_tre_after = float(tre_df["tre_after_mm"].mean())
    median_tre_after = float(tre_df["tre_after_mm"].median())
    max_tre_after = float(tre_df["tre_after_mm"].max())
    tre_improved = bool(mean_tre_after <= mean_tre_before)
else:
    mean_tre_before = median_tre_before = max_tre_before = None
    mean_tre_after = median_tre_after = max_tre_after = None
    tre_improved = False

print("validation_source:", VALIDATION_SOURCE)
print(tre_df)
print("mean_TRE_before:", mean_tre_before, "mean_TRE_after:", mean_tre_after, "improved:", tre_improved)

GATE_D_independent_validation = (
    "PASS" if (VALIDATION_SOURCE != "unavailable" and tre_improved)
    else "PARTIAL" if VALIDATION_SOURCE != "unavailable"
    else "FAIL"
)
print("GATE_D_independent_validation:", GATE_D_independent_validation)

fig, ax = plt.subplots(figsize=(6, 4))
if len(tre_df):
    x = np.arange(len(tre_df))
    width = 0.35
    ax.bar(x - width / 2, tre_df["tre_before_mm"], width, label="before")
    ax.bar(x + width / 2, tre_df["tre_after_mm"], width, label="after")
    ax.set_xticks(x)
    ax.set_xticklabels([f"lm{i}" for i in range(len(tre_df))])
    ax.set_ylabel("TRE (mm)")
    ax.legend()
    ax.set_title("Target Registration Error: before vs after (research landmarks, not clinical)")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "tre_before_after.png", dpi=120)
plt.close(fig)


validation_source: manual_research_landmarks
   fraction  tre_before_mm  tre_after_mm
0      0.15            0.0      2.027425
1      0.50            0.0      2.033314
2      0.85            0.0      2.038180
mean_TRE_before: 0.0 mean_TRE_after: 2.0329728780624663 improved: False
GATE_D_independent_validation: PARTIAL


## 14. Decisión de registración sagital

Estados posibles: `VALIDATED_FOR_TEST_STUDY`, `OPTIMIZATION_SUCCESS_VALIDATION_INSUFFICIENT`,
`REGISTRATION_FAILED`, `INVALID_GEOMETRY`.


In [17]:
if not DICOM_RECONSTRUCTION_OK:
    sagittal_decision = "INVALID_GEOMETRY"
elif GATE_C_registration_optimization != "PASS":
    sagittal_decision = "REGISTRATION_FAILED"
elif GATE_D_independent_validation == "PASS":
    sagittal_decision = "VALIDATED_FOR_TEST_STUDY"
else:
    sagittal_decision = "OPTIMIZATION_SUCCESS_VALIDATION_INSUFFICIENT"

print("Sagittal T1->T2 registration decision:", sagittal_decision)


Sagittal T1->T2 registration decision: OPTIMIZATION_SUCCESS_VALIDATION_INSUFFICIENT


## 15. Axial orientation clusters — registración individual contra Sagittal T2

Cada cluster de orientación (Sección 7) se trata **independientemente**: origin/direction/
spacing/slice-ordering propios, sin interpolar huecos entre clusters, y **sin afirmar** que
`cluster_k` corresponde a ningún nivel lumbar (`L1-L2`, etc. — eso es Notebook 67).

El registro Axial → Sagittal T2 es intrínsecamente más difícil (cobertura parcial, orientación
distinta, anisotropía, pocas slices, multimodalidad). Se permite explícitamente que algunos
clusters fallen o queden con validación insuficiente — **no se ajustan resultados
manualmente** para forzar 5 PASS.

Antes de optimizar, se calcula el solapamiento físico aproximado (bounding boxes axis-aligned)
entre el cluster y Sagittal T2; si es ~0, se marca `insufficient_overlap` sin correr el
optimizador sobre datos que no pueden converger significativamente.


In [18]:
def build_sitk_volume_from_ordered(ordered_instances: list[tuple[float, "DicomInstance"]]) -> tuple["sitk.Image", dict]:
    first_ds = ordered_instances[0][1].dataset
    row_cos, col_cos = row_column_cosines(first_ds.ImageOrientationPatient)
    if len(ordered_instances) > 1:
        p0 = np.asarray(ordered_instances[0][1].dataset.ImagePositionPatient, dtype=np.float64)
        p1 = np.asarray(ordered_instances[1][1].dataset.ImagePositionPatient, dtype=np.float64)
        slice_vec = p1 - p0
        slice_spacing = float(np.linalg.norm(slice_vec))
        slice_dir = slice_vec / slice_spacing if slice_spacing > 0 else plane_normal(row_cos, col_cos)
    else:
        slice_dir = plane_normal(row_cos, col_cos)
        slice_spacing = float(getattr(first_ds, "SpacingBetweenSlices", getattr(first_ds, "SliceThickness", 1.0)))

    pixel_spacing = first_ds.PixelSpacing
    origin = np.asarray(ordered_instances[0][1].dataset.ImagePositionPatient, dtype=np.float64)
    arrays = [load_pixel_array_for_entry(DICOM_SOURCE_PATH, inst.entry_name) for _, inst in ordered_instances]
    volume = np.stack(arrays, axis=0).astype(np.float32)

    img = sitk.GetImageFromArray(volume)
    img.SetOrigin(tuple(origin))
    img.SetSpacing((float(pixel_spacing[1]), float(pixel_spacing[0]), slice_spacing))
    direction_matrix = np.column_stack([row_cos, col_cos, slice_dir])
    img.SetDirection(tuple(direction_matrix.flatten()))
    return img, {"origin": origin.tolist(), "spacing": img.GetSpacing(), "direction": direction_matrix.flatten().tolist()}


def volume_physical_bbox(img: "sitk.Image") -> tuple[np.ndarray, np.ndarray]:
    size = img.GetSize()
    corners = []
    for i in (0, size[0] - 1):
        for j in (0, size[1] - 1):
            for k in (0, size[2] - 1):
                corners.append(np.array(img.TransformIndexToPhysicalPoint((i, j, k))))
    corners = np.array(corners)
    return corners.min(axis=0), corners.max(axis=0)


def bbox_overlap_estimate(bbox_a, bbox_b) -> float:
    lo = np.maximum(bbox_a[0], bbox_b[0])
    hi = np.minimum(bbox_a[1], bbox_b[1])
    inter = np.clip(hi - lo, 0, None)
    inter_vol = float(np.prod(inter))
    vol_a = float(np.prod(bbox_a[1] - bbox_a[0]))
    vol_b = float(np.prod(bbox_b[1] - bbox_b[0]))
    union = vol_a + vol_b - inter_vol
    return inter_vol / union if union > 0 else 0.0


axial_cluster_registration_rows = []
axial_cross_frame_transforms = []

if DICOM_RECONSTRUCTION_OK and axial_clusters_by_series:
    fixed_img_ax = sitk.Cast(sitk_volumes[sag_t2_ids[0]], sitk.sitkFloat32)
    fixed_bbox = volume_physical_bbox(fixed_img_ax)

    for sid, clusters in axial_clusters_by_series.items():
        for cluster in clusters:
            cluster_id = cluster["cluster_id"]
            ordered = cluster["ordered_instances"]
            slice_count = cluster["slice_count"]
            row_warn = []
            status = "attempted"
            metric_before = metric_after = None
            optimizer_iterations = None
            optimizer_stop_condition = None
            translation_magnitude_mm = rotation_magnitude_deg = None
            inverse_available = False
            transform_obj = None

            try:
                moving_img, meta = build_sitk_volume_from_ordered([(0.0, item["inst"]) for item in ordered])
            except Exception as exc:
                status = "reconstruction_failed"
                row_warn.append(str(exc))
                moving_img = None

            if moving_img is not None:
                moving_bbox = volume_physical_bbox(moving_img)
                overlap = bbox_overlap_estimate(fixed_bbox, moving_bbox)
                if overlap <= 1e-6:
                    status = "insufficient_overlap"
                    row_warn.append(f"bbox_overlap_estimate={overlap:.4f}")
                else:
                    try:
                        initial_t = sitk.CenteredTransformInitializer(
                            fixed_img_ax, moving_img, sitk.Euler3DTransform(),
                            sitk.CenteredTransformInitializerFilter.GEOMETRY,
                        )
                        method = sitk.ImageRegistrationMethod()
                        bins = min(50, max(8, slice_count * 4))
                        method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=bins)
                        method.SetMetricSamplingStrategy(method.RANDOM)
                        method.SetMetricSamplingPercentage(0.30, seed=2026)
                        method.SetInterpolator(sitk.sitkLinear)
                        shrink = [2, 1] if slice_count >= 4 else [1]
                        smooth = [1, 0] if slice_count >= 4 else [0]
                        method.SetShrinkFactorsPerLevel(shrink)
                        method.SetSmoothingSigmasPerLevel(smooth)
                        method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
                        method.SetOptimizerAsRegularStepGradientDescent(
                            learningRate=1.0, minStep=1e-4, numberOfIterations=150, gradientMagnitudeTolerance=1e-6,
                        )
                        method.SetOptimizerScalesFromPhysicalShift()
                        method.SetInitialTransform(initial_t, inPlace=False)

                        metric_before = method.MetricEvaluate(fixed_img_ax, moving_img)
                        transform_obj = method.Execute(fixed_img_ax, moving_img)
                        metric_after = method.GetMetricValue()
                        optimizer_stop_condition = method.GetOptimizerStopConditionDescription()
                        optimizer_iterations = method.GetOptimizerIteration()

                        euler_component = sitk.Euler3DTransform(transform_obj) if not isinstance(transform_obj, sitk.CompositeTransform) else sitk.Euler3DTransform(transform_obj.GetNthTransform(transform_obj.GetNumberOfTransforms() - 1))
                        ser = serialize_euler3d_transform(euler_component)
                        translation_magnitude_mm = ser["translation_magnitude_mm"]
                        rotation_magnitude_deg = ser["rotation_magnitude_deg"]
                        inverse_available = ser["inverse_available"]

                        if metric_after is not None and np.isfinite(metric_after):
                            status = "optimizer_success"
                        else:
                            status = "optimizer_failed"

                        # Engineering sanity thresholds (NOT clinical): flag implausibly large
                        # solutions for a small, cross-modal, anisotropic axial cluster instead of
                        # silently accepting "optimizer_success" as if it were trustworthy.
                        if translation_magnitude_mm is not None and translation_magnitude_mm > 30.0:
                            row_warn.append(f"translation_magnitude_{translation_magnitude_mm:.1f}mm_exceeds_30mm_engineering_threshold")
                        if rotation_magnitude_deg is not None and rotation_magnitude_deg > 15.0:
                            row_warn.append(f"rotation_magnitude_{rotation_magnitude_deg:.1f}deg_exceeds_15deg_engineering_threshold")

                        sitk.WriteTransform(euler_component, str(TRANSFORMS_DIR / f"axial_{sid}_cluster{cluster_id}_euler3d.tfm"))
                        axial_cross_frame_transforms.append({
                            "schemaVersion": "pfi.post-e50.cross-frame-transform.v0",
                            "studyOpaqueId": STUDY_OPAQUE_ID,
                            "sourceSeries": sid,
                            "sourceCluster": cluster_id,
                            "targetSeries": sag_t2_ids[0],
                            "targetRole": "sagittal_t2",
                            "transformType": "rigid",
                            "status": status,
                            "matrix4x4": ser["matrix_4x4"],
                            "inverseAvailable": inverse_available,
                            "validation": {"source": "unavailable", "note": "no independent axial landmark validation implemented in this run"},
                        })
                    except RuntimeError as exc:
                        status = "optimizer_failed"
                        row_warn.append(str(exc))

            axial_cluster_registration_rows.append({
                "study_opaque_id": STUDY_OPAQUE_ID,
                "cluster_id": cluster_id,
                "slice_count": slice_count,
                "fixed_role": "sagittal_t2",
                "moving_role": "axial_t2_cluster",
                "initialization_method": "geometry_centroid_initialization_cross_frame_unvalidated",
                "registration_method": "rigid_euler3d_mattes_mi",
                "metric": "mattes_mutual_information",
                "metric_before": float(metric_before) if metric_before is not None else None,
                "metric_after": float(metric_after) if metric_after is not None else None,
                "optimizer_iterations": optimizer_iterations,
                "optimizer_stop_condition": optimizer_stop_condition,
                "translation_magnitude_mm": translation_magnitude_mm,
                "rotation_magnitude_deg": rotation_magnitude_deg,
                "inverse_available": inverse_available,
                "independent_validation_source": "unavailable",
                "tre_before_mean_mm": None,
                "tre_after_mean_mm": None,
                "registration_status": status,
                "warnings": "; ".join(row_warn),
            })

axial_cluster_registration_metrics = pd.DataFrame(axial_cluster_registration_rows)
axial_cluster_registration_metrics


,study_opaque_id,cluster_id,slice_count,fixed_role,moving_role,initialization_method,registration_method,metric,metric_before,metric_after,optimizer_iterations,optimizer_stop_condition,translation_magnitude_mm,rotation_magnitude_deg,inverse_available,independent_validation_source,tre_before_mean_mm,tre_after_mean_mm,registration_status,warnings
0,206aa67ee4e6,0,6,sagittal_t2,axial_t2_cluster,geometry_centroid_initialization_cross_frame_u...,rigid_euler3d_mattes_mi,mattes_mutual_information,-0.050356,-0.039473,150,RegularStepGradientDescentOptimizerv4: Maximum...,76.401570,6.086347,True,unavailable,None,None,optimizer_success,translation_magnitude_76.4mm_exceeds_30mm_engi...
1,206aa67ee4e6,1,4,sagittal_t2,axial_t2_cluster,geometry_centroid_initialization_cross_frame_u...,rigid_euler3d_mattes_mi,mattes_mutual_information,-0.058371,-0.065701,150,RegularStepGradientDescentOptimizerv4: Maximum...,107.520069,7.882122,True,unavailable,None,None,optimizer_success,translation_magnitude_107.5mm_exceeds_30mm_eng...
2,206aa67ee4e6,2,4,sagittal_t2,axial_t2_cluster,geometry_centroid_initialization_cross_frame_u...,rigid_euler3d_mattes_mi,mattes_mutual_information,-0.086924,-0.478150,150,RegularStepGradientDescentOptimizerv4: Maximum...,2.588029,1.002177,True,unavailable,None,None,optimizer_success,
3,206aa67ee4e6,3,5,sagittal_t2,axial_t2_cluster,geometry_centroid_initialization_cross_frame_u...,rigid_euler3d_mattes_mi,mattes_mutual_information,-0.090406,-0.487274,150,RegularStepGradientDescentOptimizerv4: Maximum...,1.341962,2.474624,True,unavailable,None,None,optimizer_success,
4,206aa67ee4e6,4,5,sagittal_t2,axial_t2_cluster,geometry_centroid_initialization_cross_frame_u...,rigid_euler3d_mattes_mi,mattes_mutual_information,-0.068134,-0.121852,150,RegularStepGradientDescentOptimizerv4: Maximum...,82.031119,7.839482,True,unavailable,None,None,optimizer_success,translation_magnitude_82.0mm_exceeds_30mm_engi...


In [19]:
fig, ax = plt.subplots(figsize=(8, 4))
if len(axial_cluster_registration_metrics):
    status_counts = axial_cluster_registration_metrics["registration_status"].value_counts()
    ax.bar(status_counts.index, status_counts.values)
    ax.set_title("Axial cluster registration status summary")
    ax.set_ylabel("count")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "axial_cluster_registration_summary.png", dpi=120)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
if len(axial_cluster_registration_metrics):
    plot_df = axial_cluster_registration_metrics.dropna(subset=["metric_before", "metric_after"])
    if len(plot_df):
        x = np.arange(len(plot_df))
        width = 0.35
        ax.bar(x - width / 2, plot_df["metric_before"], width, label="before")
        ax.bar(x + width / 2, plot_df["metric_after"], width, label="after")
        ax.set_xticks(x)
        ax.set_xticklabels([f"c{c}" for c in plot_df["cluster_id"]])
        ax.set_ylabel("Mattes MI (minimized; lower/more negative = better)")
        ax.legend()
        ax.set_title("Axial cluster registration metric before/after")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "registration_metric_comparison.png", dpi=120)
plt.close(fig)
print("Axial figures written.")


Axial figures written.


## 16. Transform chain: Axial pixel → Axial patient → estimated transform → Sagittal T2

```
Axial pixel -> Axial native patient coordinate -> estimated cross-frame transform -> Sagittal T2 reference space
```

`native_point_to_reference_space()` / `reference_point_to_native_space()` implementan la cadena
completa usando el transform estimado (si existe) para ese cluster/serie. Se testea el
round-trip con puntos sintéticos: **error `< 1e-6 mm`** — esto valida la matemática del
transform (identidad numérica ida/vuelta), **no** la exactitud anatómica.


In [20]:
def native_point_to_reference_space(point_xyz, transform: "sitk.Transform") -> np.ndarray:
    inverse = transform.GetInverse()
    return np.array(inverse.TransformPoint(tuple(point_xyz)))


def reference_point_to_native_space(point_xyz, transform: "sitk.Transform") -> np.ndarray:
    return np.array(transform.TransformPoint(tuple(point_xyz)))


transform_roundtrip_errors_mm = []
if sagittal_registration_result is not None:
    t = sagittal_registration_result["final_transform"]
    rng = np.random.default_rng(2026)
    for _ in range(50):
        synthetic_point = rng.uniform(-100, 100, size=3)
        ref_space = native_point_to_reference_space(synthetic_point, t)
        back_native = reference_point_to_native_space(ref_space, t)
        err = float(np.linalg.norm(back_native - synthetic_point))
        transform_roundtrip_errors_mm.append(err)

max_transform_roundtrip_error_mm = max(transform_roundtrip_errors_mm) if transform_roundtrip_errors_mm else None
GATE_E_transform_roundtrip = "PASS" if (max_transform_roundtrip_error_mm is not None and max_transform_roundtrip_error_mm < 1e-6) else "FAIL"
print("max transform_roundtrip_error_mm:", max_transform_roundtrip_error_mm)
print("GATE_E_transform_roundtrip:", GATE_E_transform_roundtrip)


max transform_roundtrip_error_mm: 5.123796534383003e-14
GATE_E_transform_roundtrip: PASS


## 17. `CrossFrameTransform` — registro experimental

Contrato experimental (`pfi.post-e50.cross-frame-transform.v0`), **no modifica contratos
productivos**. Se serializan todos los transforms intentados (sagital T1→T2 y cada cluster
axial), con su `status` real, no solo los que "pasaron".


In [21]:
cross_frame_transforms = []

if sagittal_transform_json is not None:
    cross_frame_transforms.append({
        "schemaVersion": "pfi.post-e50.cross-frame-transform.v0",
        "studyOpaqueId": STUDY_OPAQUE_ID,
        "sourceSeries": sag_t1_ids[0],
        "sourceCluster": None,
        "targetSeries": sag_t2_ids[0],
        "targetRole": "sagittal_t2",
        "transformType": "rigid",
        "status": sagittal_decision,
        "matrix4x4": sagittal_transform_json["matrix_4x4"],
        "inverseAvailable": sagittal_transform_json["inverse_available"],
        "validation": {
            "source": VALIDATION_SOURCE,
            "meanTREBeforeMm": mean_tre_before,
            "meanTREAfterMm": mean_tre_after,
        },
    })

cross_frame_transforms.extend(axial_cross_frame_transforms)
print(f"CrossFrameTransform records: {len(cross_frame_transforms)}")


CrossFrameTransform records: 6


## 18. Quality gates A–H


In [22]:
gate_status: dict[str, str] = {}
gate_status["GATE_A_dicom_reconstruction"] = GATE_A_dicom_reconstruction
gate_status["GATE_B_simpleitk_parity"] = GATE_B_orientation
gate_status["GATE_C_sagittal_registration_optimization"] = GATE_C_registration_optimization
gate_status["GATE_D_sagittal_independent_validation"] = GATE_D_independent_validation
gate_status["GATE_E_transform_roundtrip"] = GATE_E_transform_roundtrip

if len(axial_cluster_registration_metrics) == 0:
    gate_status["GATE_F_axial_methodology_executable"] = "FAIL"
elif (axial_cluster_registration_metrics["registration_status"] == "reconstruction_failed").all():
    gate_status["GATE_F_axial_methodology_executable"] = "FAIL"
else:
    gate_status["GATE_F_axial_methodology_executable"] = "PASS"

any_axial_optimizer_success = bool((axial_cluster_registration_metrics["registration_status"] == "optimizer_success").any()) if len(axial_cluster_registration_metrics) else False
any_axial_independently_validated = False  # not implemented in this run -- see Section 13 note on scope
if any_axial_independently_validated:
    gate_status["GATE_G_axial_independent_validation"] = "PASS"
elif any_axial_optimizer_success:
    gate_status["GATE_G_axial_independent_validation"] = "PARTIAL"
else:
    gate_status["GATE_G_axial_independent_validation"] = "FAIL"

gate_status["GATE_H_privacy"] = None  # evaluated below

gates_df = pd.DataFrame(sorted(gate_status.items()), columns=["gate", "status"])
gates_df


,gate,status
0,GATE_A_dicom_reconstruction,PASS
1,GATE_B_simpleitk_parity,FAIL
2,GATE_C_sagittal_registration_optimization,PASS
3,GATE_D_sagittal_independent_validation,PARTIAL
4,GATE_E_transform_roundtrip,PASS
5,GATE_F_axial_methodology_executable,PASS
6,GATE_G_axial_independent_validation,PARTIAL
7,GATE_H_privacy,None


## 19. Privacy audit (GATE H)


In [23]:
forbidden_values = set()
if REAL_DICOM_USED:
    for inst in instances:
        ds = inst.dataset
        for field_name in FORBIDDEN_IDENTIFIER_FIELDS:
            value = getattr(ds, field_name, None)
            if value:
                forbidden_values.add(str(value))
safe_write_text._forbidden_values = forbidden_values

candidate_outputs = {
    "sagittal_transform": json.dumps(sagittal_transform_json, default=str) if sagittal_transform_json else "",
    "axial_cluster_registration_metrics": axial_cluster_registration_metrics.to_csv(index=False) if len(axial_cluster_registration_metrics) else "",
    "cross_frame_transforms": json.dumps(cross_frame_transforms, default=str),
    "tre_df": tre_df.to_csv(index=False) if len(tre_df) else "",
    "simpleitk_geometry_residuals": simpleitk_geometry_residuals.to_csv(index=False) if len(simpleitk_geometry_residuals) else "",
    "slice_orientation_deviation": "\n".join(df.to_csv(index=False) for df in slice_orientation_deviation_by_series.values()),
}

privacy_findings = []
for name, text in candidate_outputs.items():
    for value in forbidden_values:
        if value in text:
            privacy_findings.append(f"{name} contains a raw forbidden identifier value")
    if "C:\\Users\\" in text or "/Users/" in text:
        privacy_findings.append(f"{name} contains a local filesystem path")

GATE_H_PRIVACY_PASS = len(privacy_findings) == 0
gate_status["GATE_H_privacy"] = "PASS" if GATE_H_PRIVACY_PASS else "FAIL"
gates_df.loc[gates_df["gate"] == "GATE_H_privacy", "status"] = gate_status["GATE_H_privacy"]

print("Privacy findings:", privacy_findings if privacy_findings else "(none)")
print("GATE_H_privacy:", gate_status["GATE_H_privacy"])
if not GATE_H_PRIVACY_PASS:
    warnings.append(f"Privacy audit found issues: {privacy_findings}")

print(gates_df)


Privacy findings: (none)
GATE_H_privacy: PASS
                                        gate   status
0                GATE_A_dicom_reconstruction     PASS
1                    GATE_B_simpleitk_parity     FAIL
2  GATE_C_sagittal_registration_optimization     PASS
3     GATE_D_sagittal_independent_validation  PARTIAL
4                 GATE_E_transform_roundtrip     PASS
5        GATE_F_axial_methodology_executable     PASS
6        GATE_G_axial_independent_validation  PARTIAL
7                             GATE_H_privacy     PASS


## 20. Decisión general y `ready_for_level_localization`

`CROSS_FRAME_REGISTRATION_VALIDATED_FOR_TEST_STUDY` exige A–H todos `PASS`. Si la optimización
funciona pero falta validación independiente suficiente (típicamente `GATE_G`, dado que no se
implementó validación independiente para clusters axiales en esta corrida — ver limitaciones):
`PARTIAL`. No se promueve automáticamente a validado.


In [24]:
def overall_gate_status(statuses: list[str]) -> str:
    if any(s == "FAIL" for s in statuses):
        return "FAIL"
    if any(s == "PARTIAL" for s in statuses):
        return "PARTIAL"
    return "PASS"


QUALITY_GATE_OVERALL = overall_gate_status(list(gate_status.values()))

if QUALITY_GATE_OVERALL == "PASS":
    overall_decision = "CROSS_FRAME_REGISTRATION_VALIDATED_FOR_TEST_STUDY"
elif QUALITY_GATE_OVERALL == "PARTIAL":
    overall_decision = "PARTIAL"
else:
    overall_decision = "BLOCKED"

READY_FOR_LEVEL_LOCALIZATION = bool(overall_decision == "CROSS_FRAME_REGISTRATION_VALIDATED_FOR_TEST_STUDY")
blocking_gates = [g for g, s in gate_status.items() if s != "PASS"]

print("QUALITY_GATE_OVERALL:", QUALITY_GATE_OVERALL)
print("Decision:", overall_decision)
print("ready_for_level_localization:", READY_FOR_LEVEL_LOCALIZATION)
print("Blocking gates:", blocking_gates)


QUALITY_GATE_OVERALL: FAIL
Decision: BLOCKED
ready_for_level_localization: False
Blocking gates: ['GATE_B_simpleitk_parity', 'GATE_D_sagittal_independent_validation', 'GATE_G_axial_independent_validation']


## 20b. Dos conceptos de "readiness" distintos

`BLOCKED` en este notebook significa concretamente:
**`MULTIPLANAR CROSS-FRAME REGISTRATION NOT VALIDATED`** — no significa que la localización de
niveles sobre Sagittal T2 en solitario no pueda continuar. Se separan explícitamente dos
conceptos que el `ready_for_level_localization` único anterior confundía:

- **`ready_for_sagittal_level_localization`**: ¿la geometría de una sola serie sagital (Sagittal
  T2) sigue siendo válida y suficiente para que un futuro notebook intente localizar niveles
  lumbares (`L1-L2`...`L5-S1`) y sus centroides/ROIs discales **dentro de esa misma serie**? Esto
  depende de la geometría intra-serie (GATE A de Notebook 66: orientación, slice ordering,
  transforms pixel↔patient), que sigue **validada** independientemente del resultado de GATE B
  de este notebook (que es sobre la aproximación de grilla regular de SimpleITK, no sobre la
  geometría nativa por slice).
- **`ready_for_multiplanar_level_pairing`**: ¿existe evidencia suficiente para relacionar niveles
  identificados en Sagittal T2 con Sagittal T1 y con los 5 clusters axiales en un espacio de
  referencia común? Esto sigue **sin resolverse** (GATE B, D y G no en PASS).


In [25]:
READY_FOR_SAGITTAL_LEVEL_LOCALIZATION = bool(DICOM_RECONSTRUCTION_OK and gate_status["GATE_A_dicom_reconstruction"] == "PASS")
READY_FOR_MULTIPLANAR_LEVEL_PAIRING = bool(QUALITY_GATE_OVERALL == "PASS")

RESEARCH_OUTCOME = "CROSS_FRAME_REGISTRATION_BASELINE_ESTABLISHED"

print("ready_for_sagittal_level_localization:", READY_FOR_SAGITTAL_LEVEL_LOCALIZATION)
print("ready_for_multiplanar_level_pairing:", READY_FOR_MULTIPLANAR_LEVEL_PAIRING)
print("research_outcome:", RESEARCH_OUTCOME)
print()
print("Overall Notebook 66B decision (unchanged):", overall_decision)
print("Meaning: MULTIPLANAR CROSS-FRAME REGISTRATION NOT VALIDATED (not: sagittal localization cannot continue).")


ready_for_sagittal_level_localization: True
ready_for_multiplanar_level_pairing: False
research_outcome: CROSS_FRAME_REGISTRATION_BASELINE_ESTABLISHED

Overall Notebook 66B decision (unchanged): BLOCKED
Meaning: MULTIPLANAR CROSS-FRAME REGISTRATION NOT VALIDATED (not: sagittal localization cannot continue).


## 21. Artefactos de salida

Todo se escribe exclusivamente dentro de `artifacts/post_e50/cross_frame_registration/` y
`reports/post_e50/`. Sin DICOM, sin arrays de píxeles crudos, sin identificadores sensibles.


In [26]:
def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)


registration_environment = {
    "generated_at": GENERATED_AT,
    "simpleitk_version": sitk.Version_VersionString(),
    "metric": "mattes_mutual_information",
    "sagittal_config": {
        "number_of_histogram_bins": 50,
        "sampling_strategy": "RANDOM",
        "sampling_percentage": 0.20,
        "sampling_seed": 2026,
        "shrink_factors_per_level": [4, 2, 1],
        "smoothing_sigmas_per_level": [2, 1, 0],
        "optimizer": "RegularStepGradientDescent",
        "optimizer_scales": "physical_shift",
    },
}
safe_write_text(REG_DIR / "registration_environment.json", json.dumps(registration_environment, indent=2, default=_json_default, ensure_ascii=False))

sagittal_registration_export = {
    "generated_at": GENERATED_AT,
    "study_opaque_id": STUDY_OPAQUE_ID if REAL_DICOM_USED else None,
    "fixed_role": "sagittal_t2",
    "moving_role": "sagittal_t1",
    "metric_before": sagittal_registration_result["metric_before"] if sagittal_registration_result else None,
    "metric_after": sagittal_registration_result["metric_after"] if sagittal_registration_result else None,
    "metric_improved_lower_is_better": sagittal_registration_result["metric_improved_lower_is_better"] if sagittal_registration_result else None,
    "optimizer_stop_condition": sagittal_registration_result["optimizer_stop_condition"] if sagittal_registration_result else None,
    "optimizer_iterations": sagittal_registration_result["optimizer_iterations"] if sagittal_registration_result else None,
    "transform": sagittal_transform_json,
    "validation_source": VALIDATION_SOURCE,
    "tre_before_mean_mm": mean_tre_before,
    "tre_before_median_mm": median_tre_before,
    "tre_before_max_mm": max_tre_before,
    "tre_after_mean_mm": mean_tre_after,
    "tre_after_median_mm": median_tre_after,
    "tre_after_max_mm": max_tre_after,
    "decision": sagittal_decision,
}
safe_write_text(REG_DIR / "sagittal_t1_to_t2_registration.json", json.dumps(sagittal_registration_export, indent=2, default=_json_default, ensure_ascii=False))

sagittal_registration_metrics_df = pd.DataFrame([{
    "study_opaque_id": STUDY_OPAQUE_ID if REAL_DICOM_USED else None,
    "fixed_role": "sagittal_t2", "moving_role": "sagittal_t1",
    "metric_before": sagittal_registration_export["metric_before"],
    "metric_after": sagittal_registration_export["metric_after"],
    "optimizer_iterations": sagittal_registration_export["optimizer_iterations"],
    "translation_magnitude_mm": sagittal_transform_json["translation_magnitude_mm"] if sagittal_transform_json else None,
    "rotation_magnitude_deg": sagittal_transform_json["rotation_magnitude_deg"] if sagittal_transform_json else None,
    "tre_before_mean_mm": mean_tre_before, "tre_after_mean_mm": mean_tre_after,
    "decision": sagittal_decision,
}])
sagittal_registration_metrics_df.to_csv(REG_DIR / "sagittal_registration_metrics.csv", index=False)

if len(axial_cluster_registration_metrics):
    axial_cluster_registration_metrics.to_csv(REG_DIR / "axial_cluster_registration_metrics.csv", index=False)
else:
    safe_write_text(REG_DIR / "axial_cluster_registration_metrics.csv", "no_axial_clusters_found\n")

safe_write_text(REG_DIR / "cross_frame_transforms.json", json.dumps(cross_frame_transforms, indent=2, default=_json_default, ensure_ascii=False))
safe_write_text(REG_DIR / "registration_quality_gates.json", json.dumps({"gates": gate_status, "overall_status": QUALITY_GATE_OVERALL, "warnings": warnings}, indent=2, ensure_ascii=False))

registration_summary = {
    "generated_at": GENERATED_AT,
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
    "real_dicom_used": REAL_DICOM_USED,
    "study_opaque_id": STUDY_OPAQUE_ID if REAL_DICOM_USED else None,
    "reference_space": "sagittal_t2",
    "sagittal_decision": sagittal_decision,
    "axial_clusters_detected": len(axial_cluster_sizes),
    "axial_clusters_attempted": len(axial_cluster_registration_metrics),
    "axial_clusters_optimizer_success": int((axial_cluster_registration_metrics["registration_status"] == "optimizer_success").sum()) if len(axial_cluster_registration_metrics) else 0,
    "axial_clusters_independently_validated": 0,
    "gates": gate_status,
    "gate_b_failure_interpretation": GATE_B_FAILURE_INTERPRETATION,
    "gate_b_residual_stats_mm": {
        "max_residual_mm": max_residual_mm, "mean_residual_mm": mean_residual_mm,
        "median_residual_mm": median_residual_mm, "median_slice_spacing_mm": median_slice_spacing_mm_ref,
        "max_residual_relative_to_slice_spacing": max_residual_relative_to_slice_spacing,
    },
    "slice_orientation_deviation_deg": {
        role: {
            "max_orientation_deviation_deg": float(df["slice_orientation_angle_vs_first_deg"].max()),
            "mean_orientation_deviation_deg": float(df["slice_orientation_angle_vs_first_deg"].mean()),
        }
        for role, df in slice_orientation_deviation_by_series.items()
    },
    "quality_gate_overall": QUALITY_GATE_OVERALL,
    "decision": overall_decision,
    "decision_meaning": "MULTIPLANAR_CROSS_FRAME_REGISTRATION_NOT_VALIDATED",
    "research_outcome": RESEARCH_OUTCOME,
    "ready_for_level_localization": READY_FOR_LEVEL_LOCALIZATION,
    "ready_for_sagittal_level_localization": READY_FOR_SAGITTAL_LEVEL_LOCALIZATION,
    "ready_for_multiplanar_level_pairing": READY_FOR_MULTIPLANAR_LEVEL_PAIRING,
    "blocking_gates": blocking_gates,
    "recommended_next_notebooks": [
        {"notebook": "67_postE50_level_localization_v2.ipynb", "scope": "SAGITTAL-BASED AUTOMATIC LEVEL LOCALIZATION (L1-L2..L5-S1 + disc centroids/ROIs in Sagittal T2 only)"},
        {"notebook": "67B_postE50_axial_cluster_level_pairing.ipynb", "scope": "FUTURE WORK: re-evaluate the 5 axial orientation clusters and cross-frame registration using levels/centroids from Notebook 67"},
    ],
    "warnings": warnings,
    "limitations": limitations,
}
safe_write_text(REG_DIR / "registration_summary.json", json.dumps(registration_summary, indent=2, default=_json_default, ensure_ascii=False))

if len(simpleitk_geometry_residuals):
    simpleitk_geometry_residuals.to_csv(REG_DIR / "simpleitk_geometry_residuals.csv", index=False)

for role, df in slice_orientation_deviation_by_series.items():
    df.to_csv(REG_DIR / f"slice_orientation_deviation_{role}.csv", index=False)

print("Written:")
for f in sorted(REG_DIR.rglob("*")):
    if f.is_file():
        print(" -", f.relative_to(REPO_ROOT))


Written:
 - artifacts\post_e50\cross_frame_registration\axial_cluster_registration_metrics.csv
 - artifacts\post_e50\cross_frame_registration\cross_frame_transforms.json
 - artifacts\post_e50\cross_frame_registration\figures\axial_cluster_registration_summary.png
 - artifacts\post_e50\cross_frame_registration\figures\registration_metric_comparison.png
 - artifacts\post_e50\cross_frame_registration\figures\sagittal_checkerboard_after.png
 - artifacts\post_e50\cross_frame_registration\figures\sagittal_checkerboard_before.png
 - artifacts\post_e50\cross_frame_registration\figures\sagittal_difference_after.png
 - artifacts\post_e50\cross_frame_registration\figures\sagittal_registration_after.png
 - artifacts\post_e50\cross_frame_registration\figures\sagittal_registration_before.png
 - artifacts\post_e50\cross_frame_registration\figures\tre_before_after.png
 - artifacts\post_e50\cross_frame_registration\registration_environment.json
 - artifacts\post_e50\cross_frame_registration\registratio

## 22. Limitaciones (agregado)


In [27]:
limitations.extend([
    "Validated on a single real study (this notebook's Sagittal/Axial series); no proof of generalization across scanners/protocols.",
    "No clinical validation of any kind was performed or claimed.",
    "Rigid registration only (6 DOF); no deformable/non-rigid registration attempted.",
    "Optimizer convergence (REGISTRATION SUCCESS) is explicitly distinguished from independent "
    "validation (REGISTRATION VALIDATION); a converged optimizer alone was never treated as sufficient evidence.",
    "Axial cluster coverage is partial per cluster (4-6 slices), cross-modal (T2-weighted axial vs "
    "T1/T2 sagittal), and anisotropic -- registration for these is expected to be harder and some "
    "clusters may legitimately fail or lack sufficient overlap.",
    f"Independent validation for Sagittal T1->T2 used manual_research_landmarks: 3 heuristic "
    "image-center points per series at 15%/50%/85% of physical stack depth, NOT clinically or "
    "anatomically confirmed landmarks -- research validation only, not a productive dependency.",
    "No independent validation was implemented for axial cluster registrations in this run "
    "(GATE G capped at PARTIAL at best); this is the main blocker to a full VALIDATED decision.",
    "The Sagittal T1/T2 manual_research_landmarks TRE check is methodologically weak/partially "
    "circular: both series are numerically near-identical in geometry and the landmarks use the "
    "same geometric heuristic on each series independently, so a low TRE_before is expected by "
    "construction rather than proof of independent correspondence. TRE is still reported honestly "
    "(before=0.0mm, after~2.03mm in this run) as a directional signal, not strong evidence.",
    (f"GATE B (SimpleITK/Notebook-66 parity) genuinely FAILED at {max_parity_error_mm:.4f} mm "
     "(target < 1e-3 mm), was NOT loosened retroactively. Root cause: "
     "REGULAR_GRID_APPROXIMATION_OF_SLICEWISE_DICOM_GEOMETRY -- SimpleITK represents the whole "
     "series with one regular grid (single origin/spacing/direction) while DICOM preserves "
     "per-slice geometry independently. The scalar slice-position residual measured in Section 9b "
     f"(max~{max_residual_mm:.4f} mm) is roughly an order of magnitude smaller than this full "
     "parity residual and does not explain it by itself; slice-wise orientation variation "
     "(Section 9c) is a possible contributor but is not declared as the cause without measuring "
     "it. This is NOT a claim about DICOM as a standard, and it does NOT demonstrate an error in "
     "Notebook 66's slice-native pixel<->patient equations, which never assumed a regular grid."
     ) if max_parity_error_mm is not None else "GATE B could not be computed.",
    "No automatic lumbar level naming is performed or claimed anywhere in this notebook.",
    "No pathology detection is performed or claimed.",
    "Results may not generalize to studies with different geometry, coverage, or acquisition protocols.",
])
for w in warnings:
    if w not in limitations:
        limitations.append(w)
for item in limitations:
    print("-", item)


- Validated on a single real study (this notebook's Sagittal/Axial series); no proof of generalization across scanners/protocols.
- No clinical validation of any kind was performed or claimed.
- Rigid registration only (6 DOF); no deformable/non-rigid registration attempted.
- Optimizer convergence (REGISTRATION SUCCESS) is explicitly distinguished from independent validation (REGISTRATION VALIDATION); a converged optimizer alone was never treated as sufficient evidence.
- Axial cluster coverage is partial per cluster (4-6 slices), cross-modal (T2-weighted axial vs T1/T2 sagittal), and anisotropic -- registration for these is expected to be harder and some clusters may legitimately fail or lack sufficient overlap.
- Independent validation for Sagittal T1->T2 used manual_research_landmarks: 3 heuristic image-center points per series at 15%/50%/85% of physical stack depth, NOT clinically or anatomically confirmed landmarks -- research validation only, not a productive dependency.
- No in

## 23. Reporte (Markdown)


In [28]:
report_lines = []
report_lines.append("# Post-E50 Cross-Frame Registration")
report_lines.append("")
report_lines.append("## Objective")
report_lines.append("")
report_lines.append(
    "Evaluate whether Sagittal T1, Sagittal T2 and the Axial T2 orientation clusters -- found by "
    "Notebook 66 to have different FrameOfReferenceUID values and no explicit DICOM registration "
    "object between them -- can be related via a reproducible, validated rigid transform."
)
report_lines.append("")
report_lines.append("## Why registration is required")
report_lines.append("")
report_lines.append(
    "Direct DICOM coordinate comparison across series with different FrameOfReferenceUID is not "
    "guaranteed to be physically meaningful. Notebook 66 confirmed this study's three series lack "
    "both a matching FrameOfReference and any explicit Spatial/Deformable Registration object, "
    "leaving cross-series spatial relationship at PARTIAL. This notebook attempts to resolve that "
    "gap with an estimated, independently-checked rigid transform."
)
report_lines.append("")
report_lines.append("## Notebook 66 findings")
report_lines.append("")
report_lines.append(
    "Series discovery, within-series geometry, pixel/patient transforms and privacy audit all "
    f"PASSed. Axial T2 had {len(axial_cluster_sizes)} real orientation clusters (sizes "
    f"{axial_cluster_sizes}). All 3 series had different FrameOfReferenceUID; no registration "
    "object was found; cross-series spatial relationship was PARTIAL."
)
report_lines.append("")
report_lines.append("## Reference space")
report_lines.append("")
report_lines.append(
    "Sagittal T2 was chosen as fixed/reference space as an experimental engineering decision "
    "(central to the segmentation/localization pipeline, intra-series geometry already validated "
    "in Notebook 66) -- no clinical superiority is claimed."
)
report_lines.append("")
report_lines.append("## DICOM reconstruction")
report_lines.append("")
report_lines.append(f"- GATE A (reconstruction): `{gate_status['GATE_A_dicom_reconstruction']}`")
report_lines.append(f"- GATE B (SimpleITK/Notebook-66 physical-space parity): `{gate_status['GATE_B_simpleitk_parity']}` "
                     f"(max error observed: `{max_parity_error_mm}` mm, target `< 1e-3 mm`, NOT relaxed retroactively)")
report_lines.append(f"- gate_b_failure_interpretation: `{GATE_B_FAILURE_INTERPRETATION}`")
report_lines.append(
    "- Root cause, precisely stated (second revision): SimpleITK represents the whole stack with "
    "one regular grid (single origin/spacing/direction) while DICOM preserves geometry "
    "slice-by-slice. The scalar slice-position residual measured below is roughly an order of "
    "magnitude smaller than the full parity residual and does **not** explain it by itself; "
    "slice-wise orientation variation is a possible contributor but is not declared as the cause "
    "without measuring it (see the orientation-deviation table further below). This is **not** a "
    "general claim about the DICOM standard, and it does **not** demonstrate an error in "
    "Notebook 66's slice-native pixel<->patient equations (which never assumed a regular grid)."
)
report_lines.append("")
report_lines.append("### GATE B residual audit (`simpleitk_geometry_residuals`)")
report_lines.append("")
report_lines.append(simpleitk_geometry_residuals.to_markdown(index=False) if len(simpleitk_geometry_residuals) else "_Not computed._")
report_lines.append("")
report_lines.append(f"- max_residual_mm: `{max_residual_mm}`")
report_lines.append(f"- mean_residual_mm: `{mean_residual_mm}`")
report_lines.append(f"- median_residual_mm: `{median_residual_mm}`")
report_lines.append(f"- median_slice_spacing_mm: `{median_slice_spacing_mm_ref}`")
report_lines.append(f"- max_residual_relative_to_slice_spacing (engineering metric, not clinical): `{max_residual_relative_to_slice_spacing}`")
report_lines.append(f"- For comparison, full parity residual (GATE B above): `{max_parity_error_mm}` mm -- "
                     "roughly an order of magnitude larger than the scalar slice-position residual, "
                     "left unexplained by position non-uniformity alone.")
report_lines.append("")
report_lines.append("### Slice-wise orientation deviation (`slice_orientation_angle_vs_first_deg`, analytical only)")
report_lines.append("")
for role, df in slice_orientation_deviation_by_series.items():
    report_lines.append(f"**{role}**: max_orientation_deviation_deg = "
                         f"`{float(df['slice_orientation_angle_vs_first_deg'].max())}`, "
                         f"mean_orientation_deviation_deg = `{float(df['slice_orientation_angle_vs_first_deg'].mean())}`")
    report_lines.append("")
    report_lines.append(df.to_markdown(index=False))
    report_lines.append("")
report_lines.append(
    "This measurement is presented as-is, without declaring it the cause of the GATE B parity "
    "residual -- it is a possible contributor, offered for a future notebook to investigate "
    "further if cross-frame/multiplanar work resumes."
)
report_lines.append("")
report_lines.append("## SimpleITK configuration")
report_lines.append("")
report_lines.append(f"```json\n{json.dumps(registration_environment, indent=2, default=str)}\n```")
report_lines.append("")
report_lines.append("## Sagittal T1 to T2 registration")
report_lines.append("")
report_lines.append(f"- metric_before: `{sagittal_registration_export['metric_before']}`")
report_lines.append(f"- metric_after: `{sagittal_registration_export['metric_after']}` "
                     f"(Mattes MI minimized; lower/more negative = better)")
report_lines.append(f"- optimizer_stop_condition: `{sagittal_registration_export['optimizer_stop_condition']}`")
report_lines.append(f"- translation_magnitude_mm: `{sagittal_transform_json['translation_magnitude_mm'] if sagittal_transform_json else None}`")
report_lines.append(f"- rotation_magnitude_deg: `{sagittal_transform_json['rotation_magnitude_deg'] if sagittal_transform_json else None}`")
report_lines.append(f"- sanity warnings: `{sagittal_transform_json['sanity_warnings'] if sagittal_transform_json else None}` (engineering thresholds, not clinical)")
report_lines.append(f"- decision: `{sagittal_decision}`")
report_lines.append("")
report_lines.append("## Independent validation")
report_lines.append("")
report_lines.append(f"- validation_source: `{VALIDATION_SOURCE}`")
report_lines.append(tre_df.to_markdown(index=False) if len(tre_df) else "_No TRE landmarks computed._")
report_lines.append(f"- mean_TRE_before: `{mean_tre_before}` mm, mean_TRE_after: `{mean_tre_after}` mm")
report_lines.append("")
report_lines.append("## Axial orientation clusters")
report_lines.append("")
report_lines.append(f"Re-detected cluster count: `{len(axial_cluster_sizes)}`, sizes: `{axial_cluster_sizes}`.")
report_lines.append("")
report_lines.append("## Axial cluster registration")
report_lines.append("")
report_lines.append(axial_cluster_registration_metrics.to_markdown(index=False) if len(axial_cluster_registration_metrics) else "_No axial cluster registration attempted._")
report_lines.append("")
report_lines.append("## Transform chain")
report_lines.append("")
report_lines.append(f"`native_point_to_reference_space` / `reference_point_to_native_space` implemented and round-trip "
                     f"tested on 50 synthetic points: max error `{max_transform_roundtrip_error_mm}` mm "
                     f"(target `< 1e-6 mm`) -- `{gate_status['GATE_E_transform_roundtrip']}`.")
report_lines.append("")
report_lines.append("## Quality gates")
report_lines.append("")
report_lines.append(gates_df.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Results")
report_lines.append("")
report_lines.append(f"- Overall quality gate: `{QUALITY_GATE_OVERALL}`")
report_lines.append(f"- Decision: `{overall_decision}`")
report_lines.append(f"- ready_for_level_localization: `{READY_FOR_LEVEL_LOCALIZATION}`")
report_lines.append(f"- Blocking gates: `{blocking_gates}`")
report_lines.append("")
report_lines.append("## Failures")
report_lines.append("")
if len(axial_cluster_registration_metrics):
    failed = axial_cluster_registration_metrics[axial_cluster_registration_metrics["registration_status"] != "optimizer_success"]
    if len(failed):
        report_lines.append(f"{len(failed)} of {len(axial_cluster_registration_metrics)} axial cluster registrations did not reach `optimizer_success` "
                             f"(statuses: {failed['registration_status'].value_counts().to_dict()}). This is expected and was not manually corrected.")
    else:
        report_lines.append("All attempted axial cluster registrations reached optimizer_success (independent validation still pending -- see GATE G).")
else:
    report_lines.append("No axial cluster registration was attempted.")
report_lines.append("")
report_lines.append("## Limitations")
report_lines.append("")
for item in limitations:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## What 66B proves")
report_lines.append("")
report_lines.append(
    f"> A reproducible rigid registration pipeline (SimpleITK, Mattes MI, multi-resolution) was "
    f"built and executed end-to-end for Sagittal T1 -> Sagittal T2 on this real study, with "
    f"physical-space parity against Notebook 66's independent geometry math (GATE B), a "
    f"mathematically exact transform round-trip (GATE E), and a directional (before/after) "
    f"independent sanity check via heuristic research landmarks. Sagittal registration decision: "
    f"`{sagittal_decision}`."
)
report_lines.append("")
report_lines.append("## What 66B does not prove")
report_lines.append("")
for item in [
    "no clinical validation",
    "no proof the rigid model is sufficient (only rigid was tried; no deformable)",
    "optimizer convergence alone is not treated as validation anywhere in this notebook",
    "axial cluster registrations have no independent validation in this run -- GATE G is at best PARTIAL",
    "no automatic lumbar level naming or pathology detection",
    "single real study -- no generalization claim",
]:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Ready for Notebook 67?")
report_lines.append("")
report_lines.append(
    f"Research outcome: `{RESEARCH_OUTCOME}`. Overall Notebook 66B decision: `{overall_decision}` "
    f"(quality gate `{QUALITY_GATE_OVERALL}`) -- this means specifically "
    "`MULTIPLANAR CROSS-FRAME REGISTRATION NOT VALIDATED`, **not** that sagittal-only level "
    "localization cannot proceed."
)
report_lines.append("")
report_lines.append(f"- **Sagittal localization readiness**: `{'YES' if READY_FOR_SAGITTAL_LEVEL_LOCALIZATION else 'NO'}` "
                     "-- Sagittal T2's intra-series geometry (orientation validity, physical slice ordering, "
                     "native pixel<->patient transforms) remains validated per Notebook 66, independent of "
                     "this notebook's regular-grid GATE B residual.")
report_lines.append(f"- **Multiplanar pairing readiness**: `{'YES' if READY_FOR_MULTIPLANAR_LEVEL_PAIRING else 'NO'}` "
                     f"-- blocked by: `{blocking_gates}`. Not claimed as validated.")
report_lines.append("")
report_lines.append("### Roadmap")
report_lines.append("")
report_lines.append(
    "- **`67_postE50_level_localization_v2.ipynb`** (recommended next): scope explicitly limited to "
    "**SAGITTAL-BASED AUTOMATIC LEVEL LOCALIZATION**. Must produce L1-L2, L2-L3, L3-L4, L4-L5, L5-S1 "
    "and their disc centroids/ROIs **in Sagittal T2 only** -- no claim about axial or multiplanar "
    "correspondence."
)
report_lines.append(
    "- **`67B_postE50_axial_cluster_level_pairing.ipynb`** (future work, after 67): uses the "
    "levels/centroids produced by Notebook 67 to re-evaluate the 5 axial orientation clusters and "
    "the cross-frame registration blocked here."
)
report_lines.append("")

report_text = "\n".join(report_lines)
safe_write_text(REPORT_DIR / "post_e50_cross_frame_registration_report.md", report_text)
print(f"Report written: {(REPORT_DIR / 'post_e50_cross_frame_registration_report.md').relative_to(REPO_ROOT)}")


Report written: reports\post_e50\post_e50_cross_frame_registration_report.md


## 24. EXPERIMENT STATUS


In [29]:
EXECUTION_SECONDS = time.time() - EXECUTION_START

axial_attempted = len(axial_cluster_registration_metrics)
axial_independently_validated = 0  # see GATE G / limitations

status_block = f'''EXPERIMENT STATUS

Experiment:
Post-E50 Cross-Frame Registration

Training performed:
NO

Model modification:
NO

Checkpoint modification:
NO

Frozen source modified:
NO

DICOM copied into repository:
NO

Branch:
{GIT_BRANCH}

Notebook:
66B_postE50_cross_frame_registration.ipynb

Reference space:
sagittal_t2

Sagittal T1 -> T2:
{sagittal_decision}

Axial clusters detected:
{len(axial_cluster_sizes)}

Axial registrations attempted:
{axial_attempted}

Axial registrations independently validated:
{axial_independently_validated}

Transform roundtrip:
{gate_status["GATE_E_transform_roundtrip"]}

Privacy:
{gate_status["GATE_H_privacy"]}

GATE B failure interpretation:
{GATE_B_FAILURE_INTERPRETATION}

GATE B scalar residual stats (mm):
max={max_residual_mm} mean={mean_residual_mm} median={median_residual_mm} median_slice_spacing={median_slice_spacing_mm_ref}

GATE B full parity residual (mm, for comparison):
{max_parity_error_mm}

Slice orientation deviation (deg, analytical, not declared as sole cause):
{" | ".join(f"{role}: max={float(df['slice_orientation_angle_vs_first_deg'].max()):.6f} mean={float(df['slice_orientation_angle_vs_first_deg'].mean()):.6f}" for role, df in slice_orientation_deviation_by_series.items())}

Overall quality gate:
{QUALITY_GATE_OVERALL}

Decision:
{overall_decision}
(meaning: MULTIPLANAR CROSS-FRAME REGISTRATION NOT VALIDATED)

Research outcome:
{RESEARCH_OUTCOME}

Sagittal localization readiness:
{"YES" if READY_FOR_SAGITTAL_LEVEL_LOCALIZATION else "NO"}

Multiplanar pairing readiness:
{"YES" if READY_FOR_MULTIPLANAR_LEVEL_PAIRING else "NO"}

Blocking gates:
{blocking_gates if blocking_gates else "[]"}

Recommended next:
67_postE50_level_localization_v2.ipynb (scope: SAGITTAL-BASED AUTOMATIC LEVEL LOCALIZATION)

Execution seconds:
{EXECUTION_SECONDS:.1f}

Warnings:
{chr(10).join(warnings) if warnings else "(none)"}
'''

print(status_block)


EXPERIMENT STATUS

Experiment:
Post-E50 Cross-Frame Registration

Training performed:
NO

Model modification:
NO

Checkpoint modification:
NO

Frozen source modified:
NO

DICOM copied into repository:
NO

Branch:
research/post-e50-cross-frame-registration

Notebook:
66B_postE50_cross_frame_registration.ipynb

Reference space:
sagittal_t2

Sagittal T1 -> T2:
OPTIMIZATION_SUCCESS_VALIDATION_INSUFFICIENT

Axial clusters detected:
5

Axial registrations attempted:
5

Axial registrations independently validated:
0

Transform roundtrip:
PASS

Privacy:
PASS

GATE B failure interpretation:
REGULAR_GRID_APPROXIMATION_OF_SLICEWISE_DICOM_GEOMETRY

GATE B scalar residual stats (mm):
max=0.0001997582842676593 mean=8.53143378832429e-05 median=7.990718550976439e-05 median_slice_spacing=5.499994380038402

GATE B full parity residual (mm, for comparison):
0.0038199607329361764

Slice orientation deviation (deg, analytical, not declared as sole cause):
sagittal_t2: max=0.000000 mean=0.000000 | sagittal_